# 🤖 Local Coding LLM Project Generator (Qwen2.5-Coder-14B, Unity-tuned) — Kaggle Edition

Upload a spec `.md` file → this runs **Qwen2.5-Coder-14B-Instruct locally** (4-bit, single GPU,
no API keys, no external calls), automatically attaching the **Unity-specific LoRA adapter you
actually fine-tuned** (via Unsloth, on scraped/license-filtered Unity projects) if it's attached
as a Kaggle input dataset → it plans a file list, generates every file with generous, uncapped
output length, runs it through several correctness checks (syntax/compile validation, class-name
matching, structural balance, auto-retry on failure), and zips the result for download.

**Correction from the previous version of this notebook:** it had drifted to targeting
`Qwen/Qwen3-Coder-30B-A3B-Instruct` (a 30B MoE model), while the adapter that actually got trained
in the fine-tuning notebook is on `unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit` (dense, 14B,
different architecture entirely). Attaching a 14B-trained adapter to a 30B MoE base wouldn't
degrade gracefully -- it would fail on a module/shape mismatch, or worse, silently misattach. This
version targets the model that was actually trained, and adds a load-time compatibility check
(Step 3) so this specific class of bug surfaces immediately and clearly if it ever happens again,
instead of failing deep inside generation.

**Scope, read this before running:** this reliably generates **C# scripts, JSON data files,
`Packages/manifest.json`, and project settings text** — all plain text the model can actually get
right. It **does not** attempt to generate `.unity` scene files, `.prefab` files, or `.asset`
ScriptableObject instances — those are Unity's own binary/YAML serialization format with internal
GUID/fileID cross-references that an LLM writing them blind, with no Editor to validate against,
will very likely corrupt. Instead, the last generated file is always a `HUMAN_SETUP.md` that tells
you exactly what to wire up by hand in the Unity Editor to turn the generated code into a working
scene.

**Before running:** enable a GPU accelerator (Settings → Accelerator → **GPU T4 x2** is fine, but
this model only needs one) and turn Internet **On** (needed once, to download model weights from
Hugging Face).

**What's new in this version (12 additions on top of the model fix):**
1. Adapter/base-model compatibility check at load time
2. Fast structural (brace/paren/bracket) balance check right after each file generates
3. Automatic regeneration retry for any file that fails validation, with the specific problem fed back to the model
4. Class-name-must-match-filename validation for C# files (a hard Unity requirement for MonoBehaviours)
5. Duplicate / case-insensitive filename collision detection across the file plan, before generation starts
6. Structured per-file generation report (JSON), saved to disk and bundled into the zip
7. A single deterministic-vs-creative generation mode toggle, replacing scattered hardcoded temperatures
8. Auto-generated `README.md` summarizing the project, bundled into the zip
9. Final zip integrity check (every planned file present, non-empty, not suspiciously truncated)
10. Full syntax/compile validation pass over every generated `.cs` and `.json` file, run at the end against the zip's actual contents
11. Post-run summary statistics (files, lines of code, generation time, syntax pass rate)
12. A standalone "regenerate one file" utility cell, for fixing a single flagged file without rerunning everything


## Step 0 — Environment check

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"],
                      capture_output=True, text=True).stdout)
import psutil
print(f"System RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")


## Step 1 — Install dependencies

In [ ]:
import subprocess

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input"] + pkgs, check=True)

import sys
pip_install(["-U", "transformers>=4.45", "accelerate>=1.0", "bitsandbytes>=0.44", "sentencepiece", "peft>=0.13"])

# Best-effort install for the final C# syntax-checking step (Step 8) -- wrapped in try/except
# because tree-sitter's Python bindings have had breaking API changes across versions, and this
# notebook shouldn't fail to even START just because that one optional check can't install
# cleanly. Step 8 detects at run time whether this succeeded and falls back to a heuristic-only
# check (still real, just less precise) if not.
try:
    pip_install(["tree-sitter==0.21.3", "tree-sitter-languages==1.10.2"])
    print("Syntax-checker dependencies installed.")
except Exception as e:
    print(f"Note: tree-sitter install failed ({e}) -- Step 8's final check will fall back to "
          f"heuristic-only validation. Not fatal, continuing.")

print("Dependencies installed.")


## Step 2a — Working directory setup (non-blocking, always run this)

Defines `WORKDIR`/`OUTPUT_DIR`/`LOG_DIR`/`SPEC_PATH` -- needed by Steps 3, 4, 10a, and 10b. Split out from the upload widget below specifically so it can run as part of an automated sequence without blocking on a human uploading a file.

In [ ]:
import os, time, json as _json
import ipywidgets as widgets
from IPython.display import display, clear_output

WORKDIR = "/kaggle/working/llm_project_gen"
OUTPUT_DIR = os.path.join(WORKDIR, "generated_project")
LOG_DIR = os.path.join(WORKDIR, "logs")
for d in (WORKDIR, OUTPUT_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)

SPEC_PATH = os.path.join(WORKDIR, "spec.md")



## Step 3 — Load Qwen2.5-Coder-14B-Instruct (4-bit, single GPU) + attach the fine-tuned adapter

4-bit quantization via bitsandbytes. This is a dense 14B model (~8GB in 4-bit) -- it fits
comfortably on a single T4 with real headroom, no multi-GPU split needed (that split existed in
an earlier version of this notebook to fit a 30B MoE model that was never actually the one
fine-tuned -- see the correction at the top of this notebook).

**Adapter detection:** after the base model loads, this checks `/kaggle/input/*` for a dataset
containing `adapter_config.json` (the fine-tuning notebook's output). If found, it's attached via
PEFT -- but only after verifying the adapter's own recorded base model actually matches what just
loaded (new in this version -- **Feature 1**). If not found, it prints a warning and falls back to
the plain base model -- **attach that dataset as an input** (Add Input → search for your adapter
dataset name) before running.

**Weight caching, so you don't redownload every session:** this checks for a previously cached
copy attached as a Kaggle input dataset first. First run ever: no cache exists, so it downloads
fresh into `/kaggle/working/hf_model_cache/`. After this notebook finishes, click **Save Version**
(top right) to publish this run's output as a dataset. On your **next** run, before running this
cell: **Add Input → search "Notebook Output Files" → this same notebook's latest version** — the
check below will then find it automatically and skip the download entirely.


In [ ]:
import os
# Must be set before torch initializes any CUDA context -- reduces fragmentation.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch, glob
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Matches the model that was ACTUALLY fine-tuned (unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit
# in the training notebook). Loading via the plain base repo here (not the unsloth-prefixed one)
# is fine -- the LoRA adapter targets standard Qwen2 module names (q_proj/k_proj/v_proj/o_proj),
# so it attaches the same way regardless of which of the two repos supplied the base weights.
MODEL_ID = "Qwen/Qwen2.5-Coder-14B-Instruct"

# Look for a previously-cached copy attached as a Kaggle input dataset.
_cache_candidates = [
    p for p in glob.glob("/kaggle/input/*/hf_model_cache") + glob.glob("/kaggle/input/*")
    if os.path.isdir(p) and os.path.exists(os.path.join(p, "hub"))
]
if _cache_candidates:
    HF_CACHE_DIR = _cache_candidates[0]
    print(f"\u2705 Found a cached model dataset -- reusing it, no download needed: {HF_CACHE_DIR}")
else:
    HF_CACHE_DIR = "/kaggle/working/hf_model_cache"
    print(f"No cached model dataset attached -- downloading fresh into {HF_CACHE_DIR} (first run only). "
          f"Remember to 'Save Version' after this run so you can reuse it next time.")

os.environ["HF_HOME"] = HF_CACHE_DIR
os.makedirs(HF_CACHE_DIR, exist_ok=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Split across BOTH GPUs, not just GPU 0. This isn't about the model needing more total VRAM
# (weights alone are only ~8-9GB) -- it's that this GPU generation (Turing/T4) has no flash
# attention support at all, and this model's grouped-query attention (40 query heads vs 8 KV
# heads) can't use the memory-efficient kernel either -- so every long-context call (planning,
# and any continuation past a length cap) is stuck on PyTorch's "math" attention kernel, whose
# memory scales with the SQUARE of total context length. Splitting layers across two GPUs roughly
# halves the WEIGHT footprint on each card, freeing much more per-GPU headroom for that spike --
# unlike training, this is pure inference (no gradients, no optimizer state, no cross-device
# backward-pass sync), which is the well-supported, standard case for multi-GPU `device_map`.
n_gpus = torch.cuda.device_count()
print(f"{n_gpus} GPU(s) visible.")
if n_gpus >= 2:
    print("Loading model in 4-bit, split across both GPUs...")
    device_map = "auto"
    max_memory = {i: "13GiB" for i in range(n_gpus)}  # leaves ~1.5GB headroom per card so
    # accelerate doesn't plan right up to each GPU's absolute ceiling
else:
    print("Only 1 GPU visible -- loading in 4-bit on it (expect the same tight-memory behavior "
          "seen in earlier runs; a 2-GPU instance is strongly recommended for this model).")
    device_map = {"": 0}
    max_memory = None

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map=device_map,
    max_memory=max_memory,
    torch_dtype=torch.float16,
)
model.eval()
print("\u2705 Base model loaded.")
for _i in range(torch.cuda.device_count()):
    _free_b, _total_b = torch.cuda.mem_get_info(_i)
    print(f"    GPU {_i} memory right after load: {(_total_b - _free_b) / 1e9:.2f} GB used / "
          f"{_total_b / 1e9:.2f} GB total ({_free_b / 1e9:.2f} GB free for prompts + generation).")
print(f"    Note: Qwen2.5's ~152K-token vocabulary keeps the embedding/output layers in full "
      f"precision even under 4-bit quantization -- baseline usage well above a naive '14B in "
      f"4-bit ~ 8GB' estimate is expected for this specific model, not a sign of a leak.")

# --- Feature 1: Adapter/base-model compatibility check, BEFORE attaching ---
# Attaching a LoRA adapter trained on a different base model doesn't fail gracefully -- it can
# crash deep inside a forward pass, or in the worst case silently attach to modules that happen
# to share names, producing garbage output with no error at all. This checks the adapter's own
# recorded base model against what actually just loaded, and refuses to attach on a mismatch
# rather than letting that happen again.
from peft import PeftModel, PeftConfig

# Depth-agnostic: searches ANY depth under /kaggle/input, since Kaggle's actual mount
# structure varies (sometimes a dataset lands directly under /kaggle/input/<slug>/, sometimes
# grouped under an extra /kaggle/input/datasets/<owner>/<slug>/ level) -- rather than guess the
# depth again, recursive=True with ** searches all of them.
_adapter_candidates = [
    os.path.dirname(p) for p in glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
]
if not _adapter_candidates and os.path.isdir("/kaggle/input"):
    print(f"    (No adapter_config.json found anywhere under /kaggle/input. Full tree, for debugging:)")
    for _root, _dirs, _files in os.walk("/kaggle/input"):
        for _f in _files:
            print(f"      {os.path.join(_root, _f)}")

USING_TUNED_MODEL = False
if _adapter_candidates:
    ADAPTER_DIR = _adapter_candidates[0]
    _peft_cfg = PeftConfig.from_pretrained(ADAPTER_DIR)
    _adapter_base = _peft_cfg.base_model_name_or_path or ""
    # Compare the last path segment case-insensitively (handles "Qwen/X" vs "unsloth/X-bnb-4bit"
    # style naming differences between the plain and Unsloth-prefixed repos for the same model).
    _base_tail = MODEL_ID.split("/")[-1].lower()
    _adapter_tail = _adapter_base.split("/")[-1].lower().replace("-bnb-4bit", "")
    if _base_tail.replace("-instruct", "") not in _adapter_tail and _adapter_tail not in _base_tail:
        print(f"\u274c Adapter/base MISMATCH -- refusing to attach.")
        print(f"    Adapter was trained on: {_adapter_base}")
        print(f"    This notebook just loaded: {MODEL_ID}")
        print(f"    Attaching anyway would very likely crash or silently produce garbage output.")
        print(f"    Fix MODEL_ID above to match the adapter, or attach the correct adapter dataset.")
    else:
        print(f"\u2705 Adapter base ({_adapter_base}) matches the loaded model -- attaching: {ADAPTER_DIR}")
        model = PeftModel.from_pretrained(model, ADAPTER_DIR)
        model.eval()
        USING_TUNED_MODEL = True
        print("\u2705 Fine-tuned adapter attached. Generation below uses the Unity-tuned model.")
else:
    print("\u26a0\ufe0f  No LoRA adapter dataset found -- running the plain base model.")
    print("    Attach it via: Add Input -> search for your adapter dataset -> select it.")
    print("    IMPORTANT: attaching a dataset while the kernel is already running does NOT")
    print("    mount it immediately -- Kaggle only mounts /kaggle/input/ at kernel start. If")
    print("    you just attached it, use Run -> Restart Session, then Run All from the top.")
    print(f"    Currently mounted under /kaggle/input/: {os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else '(no /kaggle/input directory at all)'}")

if USING_TUNED_MODEL:
    print("\nUsing the Unity-tuned model for all generation below.")
else:
    print("\n\u26a0\ufe0f Using the base (non-tuned) model for all generation below.")


## Step 4 — Generation helper: uncapped length, auto-continues instead of truncating

A single `generate()` call still has a `max_new_tokens` ceiling per call (that's a hard technical
limit of the API, not a quality cap) — this helper detects when a response was cut off by that
ceiling rather than finishing naturally, and automatically continues generating from exactly where
it left off, appending the results, up to a generous overall cap. This is what "don't cap the
model, let it take its time" actually means in practice: no artificial shortening, just repeated
continuation until the model itself is done or the safety ceiling is hit.

In [ ]:
MAX_NEW_TOKENS_PER_CALL = 2048  # smaller per call, more continuations -- lower peak
                                 # memory per call while still reaching the same total length
MAX_CONTINUATIONS = 8          # hard safety ceiling: up to ~18K generated tokens per file

# --- Feature 7: single generation-mode toggle, replacing scattered hardcoded temperatures ---
# "deterministic" -- most reliable, most repeatable, best for getting compiling code on the
# first try. "creative" -- more varied output, somewhat higher chance of needing a retry.
GENERATION_MODE = "deterministic"  # "deterministic" or "creative"
PLANNING_TEMPERATURE = 0.1  # planning stays low-temperature regardless of mode -- consistent,
                             # parseable JSON structure matters more here than variety
FILE_GEN_TEMPERATURE = {"deterministic": 0.15, "creative": 0.6}[GENERATION_MODE]
print(f"Generation mode: {GENERATION_MODE} (file temperature = {FILE_GEN_TEMPERATURE})")

MAX_FIX_ATTEMPTS = 2  # Feature 3: how many times to regenerate a file that fails validation
                       # before giving up and leaving it flagged in the final report

import gc, re
GENERATION_LOG = os.path.join(LOG_DIR, "generation.log")

def _log(msg):
    ts = time.strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    with open(GENERATION_LOG, "a") as f:
        f.write(line + "\n")

def generate_long(system_prompt, user_prompt, max_new_tokens=MAX_NEW_TOKENS_PER_CALL,
                   max_continuations=MAX_CONTINUATIONS, temperature=0.2, seed_text="",
                   force_offload_cache=False):
    """Generate a response with no artificial length cap -- keeps continuing past each call's
    max_new_tokens ceiling until the model produces a natural stop, or the safety ceiling hits.

    seed_text: if provided, resumes from a partial result that already hit its length cap
    elsewhere (used by the batched generator below to hand off a cut-off file to a solo
    continuation instead of regenerating it from scratch)."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    full_output = ""
    start_attempt = 0
    if seed_text:
        full_output = seed_text
        messages.append({"role": "assistant", "content": seed_text})
        messages.append({"role": "user", "content":
            "Continue exactly where you left off. Do not repeat anything already written, do not "
            "restate the file or add any preamble -- just continue the raw content seamlessly."})
        start_attempt = 1

    for attempt in range(start_attempt, max_continuations + 1):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            _gen_kwargs = dict(
                max_new_tokens=max_new_tokens,
                do_sample=temperature > 0,
                temperature=max(temperature, 0.01),
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.15,  # guards against degenerate repetitive collapse
                # (e.g. output devolving into "2000000000...") -- seen once during planning
                # at low temperature; cheap insurance regardless of what triggered it
            )
            if force_offload_cache:
                # NOTE: this only reduces memory for the KV cache carried BETWEEN generation
                # steps -- it does nothing for the current step's own attention computation
                # during prefill. Kept because it still helps continuation steps, but it is NOT
                # the fix for a prefill-time OOM (that turned out to be the actual failure here).
                _gen_kwargs["cache_implementation"] = "offloaded"

            # REVERTED: a previous version of this cell tried to force the memory-efficient/
            # flash SDPA backends here. On a T4 (sm75), flash attention is unsupported entirely,
            # and the memory-efficient kernel can't handle this model's grouped-query attention
            # (40 query heads vs 8 KV heads) -- so excluding the "math" backend left NO working
            # kernel at all ("No available kernel. Aborting execution."). Math is the only backend
            # that actually works for this model on this GPU -- don't restrict away from it.
            # PLANNING_SPEC_MAX_CHARS (Step 5) is what actually addresses the memory concern here,
            # by keeping the math kernel's O(seq_len^2) attention matrix small enough to fit.
            try:
                output_ids = model.generate(**inputs, **_gen_kwargs)
            except torch.cuda.OutOfMemoryError:
                if force_offload_cache:
                    raise  # already offloading the cache -- a further retry on THIS lever won't help
                _log("  OOM during generate() -- retrying once with cache_implementation='offloaded'...")
                torch.cuda.empty_cache()
                _gen_kwargs["cache_implementation"] = "offloaded"
                output_ids = model.generate(**inputs, **_gen_kwargs)

        new_tokens = output_ids[0][input_len:]
        chunk = tokenizer.decode(new_tokens, skip_special_tokens=True)
        full_output += chunk
        hit_length_cap = (len(new_tokens) >= max_new_tokens - 1)  # computed BEFORE cleanup below

        # Release fragmented/cached memory before the next call -- without this,
        # reserved-but-unallocated memory accumulates across calls and continuations
        # until a later call fails to find a contiguous block, even with headroom free.
        del inputs, output_ids, new_tokens
        gc.collect()
        torch.cuda.empty_cache()

        if not hit_length_cap:
            _log(f"Generation finished naturally after {attempt + 1} call(s), "
                 f"{len(full_output)} chars total.")
            break

        if attempt == max_continuations:
            _log(f"\u26a0\ufe0f Hit the {max_continuations}-continuation safety ceiling -- "
                 f"stopping with {len(full_output)} chars (may be incomplete).")
            break

        _log(f"Call {attempt + 1} hit its length cap -- continuing...")
        messages.append({"role": "assistant", "content": chunk})
        messages.append({"role": "user", "content":
            "Continue exactly where you left off. Do not repeat anything already written, do not "
            "restate the file or add any preamble -- just continue the raw content seamlessly."})

    return full_output


BATCH_SIZE = 2  # files generated per batched call. Raises GPU utilization (both pipeline-split
                 # halves get more concurrent work per step) at the cost of more memory pressure --
                 # start here and watch `nvidia-smi -l 1` during a run; raise it if there's headroom,
                 # lower it back to 1 if you see OOMs again.

tokenizer.padding_side = "left"   # required for batched generation: left-padding means every row's
                                    # actual new tokens start at the same index, output_ids[:, input_len:]
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_batch(system_prompt, user_prompts, max_new_tokens=MAX_NEW_TOKENS_PER_CALL, temperature=0.15):
    """Generate for several prompts in one batched call instead of one at a time -- this is the
    actual lever for higher combined GPU utilization on a model that has to be split across both
    GPUs just to fit (a true independent full-utilization split per GPU isn't possible here, since
    a full 4-bit copy of a 32B model doesn't fit on a single 15GB T4 on its own)."""
    prompts = []
    for up in user_prompts:
        messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": up}]
        prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 0.01),
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    results = []
    for row in range(output_ids.shape[0]):
        new_tokens = output_ids[row][input_len:]
        chunk = tokenizer.decode(new_tokens, skip_special_tokens=True)
        hit_cap = (len(new_tokens) >= max_new_tokens - 1)
        results.append((chunk, hit_cap))

    del inputs, output_ids
    gc.collect()
    torch.cuda.empty_cache()
    return results


---

# 🤖 Fully automated path starts here

**For automation: run Steps 0/1/3/4 above, then continue directly below (Step 10a or 10b).** Do NOT run the manual upload/planning/generation cells further down (old Step 2 onward) as part of an automated "Run All" -- that flow is kept at the very end of this notebook now, for optional one-off manual testing only, specifically so it can never block an automated run by sitting in the middle waiting for a file upload.

---

## Step 10a — Direct git sync (recommended for generation)

Run this instead of relying on an HTTP round-trip for generation -- pulls `spec_docs.zip` from your repo, generates, and pushes the result straight back via git. No ngrok, no Actions workflow, no timeout risk for a job that can take 20-60+ minutes. Requires `GITHUB_PAT` in Kaggle Secrets (Add-ons -> Secrets) -- if missing, this cell will ask you for it directly rather than silently falling back to a placeholder.

In [ ]:
# --- Step 10a: Direct git-based generation sync (no ngrok, no GitHub Actions needed for this half) ---
# Pulls spec_docs.zip straight from your repo, generates, and pushes the result straight back --
# no HTTP tunnel, no timeout risk on a 20-60 minute job. Run this cell manually whenever you
# want to (re)generate; it's not a persistent server like Step 10b (/repair) below.

import subprocess, requests, base64, shutil

GITHUB_REPO = "NamanKumar6280/Local-Claude"  # <-- set this to your actual repo
GIT_BRANCH = "main"

def _require_secret(name, prompt_label=None):
    """Loud and explicit on purpose -- silently falling back to a dev placeholder here was
    exactly the confusing behavior that made it unclear whether the token was ever being read."""
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret(name)
        if val:
            print(f"\u2705 {name} loaded from Kaggle Secrets.")
            return val
    except Exception:
        pass
    print(f"\n{'='*70}\n{name} not found in Kaggle Secrets (Add-ons -> Secrets, top of notebook).")
    print(f"Add it there and re-run this cell -- OR paste it now just for this session:")
    print(f"{'='*70}")
    val = input(f"{prompt_label or name}: ").strip()
    if not val:
        raise RuntimeError(f"{name} is required -- cannot continue without it.")
    return val

GITHUB_PAT = _require_secret("GITHUB_PAT", "GitHub Personal Access Token (repo scope)")

CLONE_DIR = os.path.join(BASE_DIR, "repo_clone")

def sync_generate_and_push():
    repo_url = f"https://{GITHUB_PAT}@github.com/{GITHUB_REPO}.git"

    if os.path.isdir(CLONE_DIR):
        print("Pulling latest...")
        subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    else:
        print("Cloning repo...")
        subprocess.run(["git", "clone", "--branch", GIT_BRANCH, repo_url, CLONE_DIR], check=True)

    spec_zip_path = os.path.join(CLONE_DIR, "spec_docs.zip")
    if not os.path.exists(spec_zip_path):
        raise FileNotFoundError(f"No spec_docs.zip found in the repo at {spec_zip_path}. "
                                 f"Upload one to the repo root first.")

    global SPEC_TEXT
    SPEC_TEXT = _combine_docs_zip(open(spec_zip_path, "rb").read()) \
        if "_combine_docs_zip" in globals() else None
    if SPEC_TEXT is None:
        # Fallback combine logic if Step 10b's server cell hasn't been run this session
        import zipfile as _zf, io as _io
        extract_dir = os.path.join(WORKDIR, "spec_docs_sync")
        if os.path.isdir(extract_dir):
            shutil.rmtree(extract_dir)
        os.makedirs(extract_dir, exist_ok=True)
        with _zf.ZipFile(spec_zip_path) as zf:
            zf.extractall(extract_dir)
        priority = ["README.md", "Architecture.md", "ScriptsIndex.md", "ProjectVersion.txt",
                    "Packages.md", "Scenes.md", "Prefabs.md", "Dependencies.md", "CodingGuidelines.md"]
        files = {f: os.path.join(extract_dir, f) for f in os.listdir(extract_dir)
                 if f.endswith((".md", ".txt"))}
        ordered = [f for f in priority if f in files] + sorted(f for f in files if f not in priority)
        parts = []
        for fname in ordered:
            with open(files[fname], encoding="utf-8", errors="ignore") as fh:
                parts.append(f"=== {fname} ===\n\n{fh.read().strip()}")
        SPEC_TEXT = "\n\n\n".join(parts)

    print(f"Loaded spec: {len(SPEC_TEXT)} chars. Running planning + generation...")

    # Re-run the same verified pipeline cells (planning -> generation -> bootstrap -> readme ->
    # zip -> validation), same technique as the async server path -- reuses proven code, no
    # hand-retyped duplicate logic to drift out of sync.
    job_ns = globals()
    for cell_src in _PIPELINE_CELLS_IN_ORDER:
        exec(compile(cell_src, "<sync_pipeline_cell>", "exec"), job_ns)

    # Unpack the result STRAIGHT INTO THE REPO ROOT -- not a subfolder. This is the convention
    # game-ci/unity-builder and the whole workflow chain expect.
    import zipfile as _zf2
    with _zf2.ZipFile(job_ns["final_path"]) as zf:
        for d in ("Assets", "Packages", "ProjectSettings"):
            for member in zf.namelist():
                if member.startswith(d + "/") or member == d + "/":
                    zf.extract(member, CLONE_DIR)

    subprocess.run(["git", "-C", CLONE_DIR, "add", "Assets", "Packages", "ProjectSettings"], check=True)
    diff = subprocess.run(["git", "-C", CLONE_DIR, "diff", "--cached", "--quiet"])
    if diff.returncode == 0:
        print("Nothing changed -- generation produced an identical project.")
        return
    subprocess.run(["git", "-C", CLONE_DIR, "config", "user.email", "generator-bot@users.noreply.github.com"], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "config", "user.name", "generator-bot"], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "commit", "-m", "Generate project from spec_docs.zip"], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "push"], check=True)
    print("\u2705 Pushed generated project to GitHub -- this will trigger the compile workflow.")

# NOT auto-run on "Run All" -- this is an ALTERNATIVE to Step 10b's HTTP server below, not a
# companion to it. Uncomment the line below only if you specifically want the git-sync path
# instead of the workflow-triggered /generate HTTP path (which is what workflow 1 actually
# calls, and what this whole pipeline has been tested against).
# sync_generate_and_push()
print("Step 10a defined but not auto-run. Call sync_generate_and_push() manually if you want "
      "this path instead of Step 10b's HTTP server below.")


## Step 10 — Unified job server (generation + repair)

Run Steps 0/1/3/4 first (skip 2/5/6/6.5/6.6/7/8/9 -- server mode replaces that interactive flow). Exposes `/generate` and `/repair`, both async (submit -> poll `/status/<id>` -> download `/result/<id>`), both sharing the one model already loaded in Step 3 and processed strictly one-at-a-time through a single queue.

In [ ]:
# --- Unified job server: generation AND repair, one model, one queue, one tunnel ---------
# Both /generate and /repair use the SAME loaded model (Step 3 above) -- there's only one T4
# worth of VRAM, not enough for two separate model copies. Because of that, BOTH task types
# are processed by ONE worker thread through ONE queue -- calling model.generate() from two
# threads at once isn't safe, so this guarantees only one GPU operation happens at a time,
# whether it's a full project generation or a compiler-error repair.

import threading, queue, uuid, time, traceback, zipfile as _zf2, shutil as _shutil2, io, re as _re2
from flask import Flask, request, jsonify, send_file
from collections import defaultdict as _defaultdict2

MAX_NEW_TOKENS_PER_FILE = 1536  # was in the original repair-server notebook's separate
# config cell, never carried over when merging -- generate_fix() below needs this defined or
# every /repair call crashes with a NameError

# Cell 4 — prompt construction + generation + response parsing
import re

SYSTEM_PROMPT = (
    "You are an expert Unity/C# engineer fixing compiler errors in an "
    "Editor-tooling project. You are given one file's full current content "
    "and the exact compiler errors that point into it. Return ONLY the "
    "complete corrected file content inside a single ```csharp code block. "
    "No explanation before or after the code block. Preserve every method "
    "signature and class name that isn't directly implicated by an error."
)


def build_prompt(file_path, file_content, file_errors, general_instructions):
    err_lines = []
    for e in file_errors:
        hint = e.get("hint", {})
        err_lines.append(
            f"- line {e['line']}, col {e.get('column', '?')}: {e['code']} "
            f"{e['message']}" + (f"  [hint: {hint}]" if hint.get("kind") not in (None, "other") else "")
        )
    errors_block = "\n".join(err_lines)

    return (
        f"{general_instructions}\n\n"
        f"FILE: {file_path}\n\n"
        f"COMPILER ERRORS IN THIS FILE:\n{errors_block}\n\n"
        f"CURRENT FILE CONTENT:\n```csharp\n{file_content}\n```\n\n"
        f"Return the complete corrected file."
    )


CODE_BLOCK_RE = re.compile(r"```(?:csharp|cs)?\s*\n(.*?)```", re.DOTALL)


def extract_code(model_output):
    m = CODE_BLOCK_RE.search(model_output)
    if m:
        return m.group(1).strip() + "\n"
    # Model didn't fence it — fall back to the raw output, better than nothing
    return model_output.strip() + "\n"


@torch.inference_mode()
def generate_fix(file_path, file_content, file_errors, general_instructions):
    user_prompt = build_prompt(file_path, file_content, file_errors, general_instructions)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    output_ids = model.generate(
        input_ids,
        max_new_tokens=MAX_NEW_TOKENS_PER_FILE,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output_ids[0][input_ids.shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return extract_code(text)

# Cell 5 — zip in, zip out
import io
import json
import zipfile
from collections import defaultdict


def repair_project(zip_bytes, errors_report, general_instructions):
    """Returns (repaired_zip_bytes, summary_dict)."""
    errors_by_file = defaultdict(list)
    for e in errors_report.get("errors", []):
        errors_by_file[e["file"]].append(e)

    changed, failed = [], []

    src_zip = zipfile.ZipFile(io.BytesIO(zip_bytes))
    out_buf = io.BytesIO()
    out_zip = zipfile.ZipFile(out_buf, "w", zipfile.ZIP_DEFLATED)

    for info in src_zip.infolist():
        name = info.filename
        raw = src_zip.read(name)

        if name in errors_by_file:
            try:
                original_text = raw.decode("utf-8")
                fixed_text = generate_fix(
                    name, original_text, errors_by_file[name], general_instructions
                )
                out_zip.writestr(name, fixed_text)
                changed.append(name)
                continue
            except Exception as e:
                print(f"[repair_project] FAILED on {name}: {e}")
                failed.append({"file": name, "error": str(e)})
                # fall through — write the original unmodified rather than dropping the file

        out_zip.writestr(name, raw)

    out_zip.close()

    summary = {
        "changed": changed,
        "created": [],
        "deleted": [],
        "failed": failed,
        "still_needed_packages": errors_report.get("missingPackages", []),
    }
    return out_buf.getvalue(), summary

_PIPELINE_CELL_12 = "PLANNING_SYSTEM_PROMPT = \"\"\"You are a senior Unity/C# engineer turning a written technical \\\nspecification into a concrete file plan for a Unity project.\n\nOnly plan files you can write correctly as plain text with no Unity Editor available:\n- C# scripts (MonoBehaviours and plain classes)\n- JSON data files (for content that would normally be ScriptableObject assets -- a human will \\\nimport these into real ScriptableObjects later; do NOT plan .asset files yourself)\n- Packages/manifest.json\n- Plain text project settings notes\n- Exactly one HUMAN_SETUP.md as the last file, explaining what a human must do in the Unity \\\nEditor to finish wiring this into a working project (scenes, prefabs, importing the JSON data \\\ninto real ScriptableObjects, NavMesh baking, etc.)\n\nDo NOT plan .unity, .prefab, or .asset files -- those need Unity's Editor to keep GUID/fileID \\\nreferences valid and will be corrupted if hand-written as raw text.\n\nCRITICAL path rule: every \"path\" value MUST start with the literal prefix \"Assets/\" or\n\"Packages/\" -- for example \"Assets/_Project/Scripts/Data/ProductData.cs\", never just\n\"_Project/Scripts/Data/ProductData.cs\" or \"Scripts/Data/ProductData.cs\". A Unity project\nis only valid if its actual content lives inside a real Assets/ folder at the project\nroot -- omitting that prefix will produce a broken, unopenable project.\n\nOutput ONLY a JSON array, no other text, no markdown code fences. Each element:\n{\"path\": \"Assets/... or Packages/...\", \"type\": \"csharp|json|manifest|text|readme\", \\\n\"description\": \"what this file must contain, specific enough to write it correctly in isolation\"}\n\nPlan AT MOST 35 files total, even if the specification describes more scope than that -- prefer \\\nfewer, more substantial files (e.g. one Combat/ folder of related scripts as 3-4 files, not 15 \\\ntiny ones) over an exhaustive breakdown. This plan must fit in a single response with no \\\ncontinuation needed.\n\nKeep each \"description\" to ONE short, plain-language sentence (max ~20 words) describing WHAT \\\nthe file does -- e.g. \"ScriptableObject holding a product's name, price, and shelf category.\" \\\nDo NOT write actual C# code, attributes, field declarations, or syntax in the description -- that \\\nlevel of detail belongs in the file itself (Step 6 generates the real code from this description), \\\nnot in the plan. Verbose code-like descriptions are the main reason planning runs out of its \\\ntoken budget before finishing the file list.\n\"\"\"\n\nPLANNING_SPEC_MAX_CHARS = 6000  # lowered from 16000 -- your spec (15,903 chars) was UNDER the\n# old cap, so it never truncated, and still OOM'd. If attention falls back to the memory-heavy\n# \"math\" SDPA kernel (see Step 4's new kernel-forcing attempt), its memory scales roughly with\n# the SQUARE of sequence length -- cutting input length is the most reliable lever available,\n# more reliable than hoping the kernel-forcing trick above actually engages on this GPU/torch combo.\n_spec_for_planning = SPEC_TEXT[:PLANNING_SPEC_MAX_CHARS]\nif len(SPEC_TEXT) > PLANNING_SPEC_MAX_CHARS:\n    _spec_for_planning += (\"\\n\\n[...spec truncated for length -- plan based on what's above; \"\n                            \"per-file generation later still sees the plan's own 'description' field...]\")\n    print(f\"\\u26a0\\ufe0f  spec.md is {len(SPEC_TEXT)} chars -- truncated to {PLANNING_SPEC_MAX_CHARS} \"\n          f\"for the planning prompt to avoid a prefill OOM. If the plan looks incomplete, this is why; \"\n          f\"raise PLANNING_SPEC_MAX_CHARS above if you have GPU headroom to spare.\")\n\nplanning_user_prompt = f\"Specification:\\n\\n{_spec_for_planning}\\n\\nProduce the file plan JSON now.\"\n\n_log(f\"Starting planning pass... ({'Unity-tuned' if USING_TUNED_MODEL else 'BASE (untuned)'} model)\")\n# force_offload_cache removed -- the 2-GPU split in Step 3 is the real fix for memory pressure\n# now, and the offloaded-cache code path was implicated in a run that produced degenerate,\n# repetitive garbage output instead of crashing. repetition_penalty in Step 4 is a second\n# guard against that specific failure mode regardless of cause.\n# max_continuations=0 is deliberate: a continuation re-tokenizes the ENTIRE conversation so\n# far (including the full first call's output) as input to the next call -- for planning, that\n# meant a second call starting from ~5,900+ tokens of context, which is exactly what triggered\n# the OOM (not GPU count -- a single attention op still runs on one GPU either way). Better to\n# stop cleanly here and let the JSON-repair/retry logic below handle a truncated result than to\n# pay for an expensive, OOM-prone continuation. The 35-file cap above should make hitting the\n# length ceiling at all much less likely in the first place.\nplan_raw = generate_long(PLANNING_SYSTEM_PROMPT, planning_user_prompt,\n                          max_new_tokens=6000, max_continuations=0, temperature=PLANNING_TEMPERATURE)\n                          # 6000, up from 4096 -- a single (non-continuation) call at this length\n                          # did NOT OOM last run, it just ran out of budget mid-file. More headroom\n                          # directly reduces truncation risk; still bounded, still no continuation.\n\n# The model may still wrap the JSON in fences despite instructions -- strip defensively.\nplan_clean = plan_raw.strip()\nif plan_clean.startswith(\"```\"):\n    plan_clean = plan_clean.split(\"```\")[1]\n    if plan_clean.startswith(\"json\"):\n        plan_clean = plan_clean[4:]\nplan_clean = plan_clean.strip()\n\ndef _try_parse_json(text):\n    try:\n        return _json.loads(text), None\n    except _json.JSONDecodeError as e:\n        return None, e\n\nFILE_PLAN, _err = _try_parse_json(plan_clean)\n\nif FILE_PLAN is None:\n    # Common, cheaply-fixable glitches: trailing commas before ] or }, and single quotes\n    # instead of double quotes. Try each repair in turn before giving up.\n    _repaired = re.sub(r',(\\s*[\\]\\}])', r'\\1', plan_clean)  # strip trailing commas\n    FILE_PLAN, _err2 = _try_parse_json(_repaired)\n    if FILE_PLAN is None:\n        _repaired2 = _repaired.replace(\"'\", '\"')\n        FILE_PLAN, _err3 = _try_parse_json(_repaired2)\n\nif FILE_PLAN is None:\n    # Last resort: the array may just be genuinely truncated mid-object (ran out of token\n    # budget), not malformed -- salvage every COMPLETE top-level object before the cutoff and\n    # close the array, rather than discarding an otherwise-good partial plan entirely.\n    _last_complete = plan_clean.rfind(\"},\")\n    if _last_complete != -1:\n        _salvaged = plan_clean[:_last_complete + 1].rstrip().rstrip(\",\") + \"\\n]\"\n        FILE_PLAN, _err4 = _try_parse_json(_salvaged)\n        if FILE_PLAN is not None:\n            print(f\"\\u26a0\\ufe0f  Plan was truncated mid-file -- salvaged {len(FILE_PLAN)} complete \"\n                  f\"file entries before the cutoff and dropped the incomplete last one. If this is \"\n                  f\"fewer files than your spec needs, re-run this cell (sampling varies) or raise \"\n                  f\"max_new_tokens on the generate_long call above.\")\n\nif FILE_PLAN is None:\n    print(f\"\\u274c Planning JSON still invalid after repair attempts: {_err}\")\n    print(f\"\\n--- Raw model output (first 2000 chars) ---\\n{plan_raw[:2000]}\\n--- end ---\\n\")\n    print(\"Common fix: just re-run this cell -- planning is sampled, so a retry often produces \"\n          \"valid JSON even when this one didn't. If it keeps failing, lower PLANNING_TEMPERATURE \"\n          \"in this cell towards 0 for more deterministic (and more reliably well-formed) output.\")\n    raise _err\n\nprint(f\"\\n\\u2705 Planned {len(FILE_PLAN)} files:\\n\")\nfor entry in FILE_PLAN:\n    print(f\"  - {entry['path']}  ({entry['type']})\")\n\n# --- Feature 5: duplicate / case-insensitive filename collision detection ---\n# Unity projects commonly get imported on both case-sensitive (Linux/Mac) and case-insensitive\n# (Windows) filesystems. Two planned files that differ only by case (e.g. \"PlayerData.cs\" and\n# \"playerdata.cs\") would silently overwrite one another on Windows, or worse, produce genuinely\n# undefined behavior on Unity's asset database. Catch that here, before any generation happens,\n# rather than discovering it as a mysteriously missing file later.\n_seen_lower = {}\n_collisions = []\nfor entry in FILE_PLAN:\n    key = entry[\"path\"].lower()\n    if key in _seen_lower:\n        _collisions.append((_seen_lower[key], entry[\"path\"]))\n    else:\n        _seen_lower[key] = entry[\"path\"]\n\nif _collisions:\n    print(f\"\\n\\u26a0\\ufe0f  {len(_collisions)} case-insensitive filename collision(s) found in the plan:\")\n    for a, b in _collisions:\n        print(f\"    {a!r}  <->  {b!r}\")\n    print(\"    These will overwrite each other on case-insensitive filesystems (Windows). \"\n          \"Consider editing FILE_PLAN by hand before continuing, or re-running this cell.\")\nelse:\n    print(\"\\n\\u2705 No filename collisions in the plan.\")\n"

_PIPELINE_CELL_14 = "GENERATION_SYSTEM_PROMPT = \"\"\"You are a senior Unity/C# engineer. Write ONE complete, real, \\\ncompiling file -- never a stub, never a placeholder, never a TODO, never truncated. Take as much \\\nspace as the file genuinely needs to be correct and complete; do not artificially shorten it. \\\nOutput ONLY the raw file content -- no markdown code fences, no explanation before or after, no \\\ncommentary. If writing C#, include using statements, full method bodies, and real logic matching \\\nthe description exactly.\"\"\"\n\ndef _normalize_path(entry):\n    p = entry[\"path\"].lstrip(\"/\")\n    if not (p.startswith(\"Assets/\") or p.startswith(\"Packages/\")):\n        p = f\"Assets/{p}\"\n        entry[\"path\"] = p\n    return p\n\ndef _build_user_prompt(entry, extra_instruction=None):\n    SPEC_EXCERPT_MAX_CHARS = 4000\n    spec_for_prompt = SPEC_TEXT[:SPEC_EXCERPT_MAX_CHARS]\n    if len(SPEC_TEXT) > SPEC_EXCERPT_MAX_CHARS:\n        spec_for_prompt += \"\\n\\n[...spec truncated for length -- see the description field below for this file's specific requirements...]\"\n    prompt = (\n        f\"Specification excerpt, for background context:\\n\\n{spec_for_prompt}\\n\\n\"\n        f\"---\\n\\nNow write this specific file in full:\\n\\n\"\n        f\"Path: {entry['path']}\\nType: {entry['type']}\\nRequired contents: {entry['description']}\\n\\n\"\n        f\"Write the complete file content now.\"\n    )\n    if extra_instruction:\n        prompt += f\"\\n\\nIMPORTANT -- fix this specific problem from the previous attempt: {extra_instruction}\"\n    return prompt\n\ndef _clean_fences(content):\n    content_clean = content.strip()\n    if content_clean.startswith(\"```\"):\n        lines = content_clean.split(\"\\n\")\n        if lines[0].startswith(\"```\"):\n            lines = lines[1:]\n        if lines and lines[-1].strip() == \"```\":\n            lines = lines[:-1]\n        content_clean = \"\\n\".join(lines)\n    return content_clean\n\n# --- Feature 2: fast structural balance check (catches truncation/malformed output cheaply) ---\ndef _check_balance(content):\n    problems = []\n    pairs = [(\"{\", \"}\"), (\"(\", \")\"), (\"[\", \"]\")]\n    for open_ch, close_ch in pairs:\n        o, c = content.count(open_ch), content.count(close_ch)\n        if o != c:\n            problems.append(f\"unbalanced '{open_ch}{close_ch}': {o} open vs {c} close\")\n    return problems\n    # Note: this is a heuristic, not a real parser -- a brace character inside a string literal\n    # or comment can occasionally cause a false positive. Step 8's final pass uses a real parser\n    # where available and is the authoritative check; this one exists to catch the common,\n    # cheap-to-detect case (truncated/cut-off generation) immediately, without waiting until\n    # the whole project is done.\n\n# --- Feature 4: class-name-must-match-filename check (hard Unity requirement for MonoBehaviours) ---\n_CLASS_RE = re.compile(r'public\\s+(?:sealed\\s+|abstract\\s+|partial\\s+)*(?:class|struct|interface)\\s+(\\w+)')\n\ndef _check_class_name(entry, content):\n    if entry[\"type\"] != \"csharp\":\n        return []\n    filename_stem = os.path.splitext(os.path.basename(entry[\"path\"]))[0]\n    matches = _CLASS_RE.findall(content)\n    if len(matches) == 1 and matches[0] != filename_stem:\n        return [f\"public type '{matches[0]}' doesn't match filename '{filename_stem}' -- \"\n                f\"Unity requires these to match exactly for MonoBehaviours to attach in the Editor\"]\n    return []\n\ndef _validate_file(entry, content):\n    return _check_balance(content) + _check_class_name(entry, content)\n\ndef _write_file(entry, content):\n    out_path = os.path.join(OUTPUT_DIR, entry[\"path\"])\n    os.makedirs(os.path.dirname(out_path), exist_ok=True)\n    content_clean = _clean_fences(content)\n    with open(out_path, \"w\", encoding=\"utf-8\") as f:\n        f.write(content_clean)\n    _log(f\"  -> {entry['path']}: {len(content_clean)} chars written.\")\n    return content_clean\n\n# --- Feature 6: structured per-file generation report ---\nGENERATION_REPORT_PATH = os.path.join(LOG_DIR, \"generation_report.json\")\nGENERATION_REPORT = {}\n\ndef _generate_with_retries(entry, base_prompt, initial_content, initial_hit_cap):\n    \"\"\"Handles the length-cap-continuation case (existing behavior) AND validation-driven\n    retries (Feature 3) -- regenerating a file up to MAX_FIX_ATTEMPTS times, feeding the\n    specific detected problem back to the model each time, before giving up and flagging it.\"\"\"\n    if initial_hit_cap:\n        content = generate_long(GENERATION_SYSTEM_PROMPT, base_prompt,\n                                 max_new_tokens=MAX_NEW_TOKENS_PER_CALL,\n                                 max_continuations=MAX_CONTINUATIONS, temperature=FILE_GEN_TEMPERATURE,\n                                 seed_text=initial_content)\n    else:\n        content = initial_content\n\n    attempts = 1\n    issues = _validate_file(entry, _clean_fences(content))\n    while issues and attempts <= MAX_FIX_ATTEMPTS:\n        _log(f\"  {entry['path']}: validation failed ({'; '.join(issues)}) -- retry {attempts}/{MAX_FIX_ATTEMPTS}...\")\n        retry_prompt = _build_user_prompt(entry, extra_instruction=\"; \".join(issues))\n        content = generate_long(GENERATION_SYSTEM_PROMPT, retry_prompt,\n                                 max_new_tokens=MAX_NEW_TOKENS_PER_CALL,\n                                 max_continuations=MAX_CONTINUATIONS, temperature=FILE_GEN_TEMPERATURE)\n        issues = _validate_file(entry, _clean_fences(content))\n        attempts += 1\n\n    status = \"ok\" if attempts == 1 and not issues else (\"retried_ok\" if not issues else \"failed\")\n    GENERATION_REPORT[entry[\"path\"]] = {\"status\": status, \"attempts\": attempts, \"issues\": issues}\n    if issues:\n        _log(f\"  \\u26a0\\ufe0f {entry['path']}: still failing validation after {attempts} attempt(s): {'; '.join(issues)}\")\n    return content\n\n# Skip anything already generated (checkpoint/resume behavior across session restarts).\npending = []\nfor entry in FILE_PLAN:\n    _normalize_path(entry)\n    out_path = os.path.join(OUTPUT_DIR, entry[\"path\"])\n    if os.path.exists(out_path):\n        print(f\"Skipping (already generated): {entry['path']}\")\n        GENERATION_REPORT.setdefault(entry[\"path\"], {\"status\": \"ok\", \"attempts\": 0, \"issues\": []})\n    else:\n        pending.append(entry)\n\nprint(f\"\\n{len(pending)} file(s) to generate, {len(FILE_PLAN) - len(pending)} already done.\\n\")\n\n_gen_start_time = time.time()\n\nfor batch_start in range(0, len(pending), BATCH_SIZE):\n    batch = pending[batch_start: batch_start + BATCH_SIZE]\n    _log(f\"Generating batch of {len(batch)}: {[e['path'] for e in batch]}\")\n\n    user_prompts = [_build_user_prompt(entry) for entry in batch]\n    batch_results = generate_batch(GENERATION_SYSTEM_PROMPT, user_prompts,\n                                    max_new_tokens=MAX_NEW_TOKENS_PER_CALL, temperature=FILE_GEN_TEMPERATURE)\n\n    for entry, prompt, (chunk, hit_cap) in zip(batch, user_prompts, batch_results):\n        content = _generate_with_retries(entry, prompt, chunk, hit_cap)\n        _write_file(entry, content)\n\n_gen_elapsed = time.time() - _gen_start_time\n\nwith open(GENERATION_REPORT_PATH, \"w\", encoding=\"utf-8\") as f:\n    _json.dump(GENERATION_REPORT, f, indent=2)\n\n_failed = [p for p, r in GENERATION_REPORT.items() if r[\"status\"] == \"failed\"]\n_retried = [p for p, r in GENERATION_REPORT.items() if r[\"status\"] == \"retried_ok\"]\nprint(f\"\\n\\u2705 All {len(FILE_PLAN)} files generated in {OUTPUT_DIR} ({_gen_elapsed:.0f}s this run)\")\nprint(f\"   {len(_retried)} needed a retry and passed after fixing; {len(_failed)} still flagged after {MAX_FIX_ATTEMPTS} attempts.\")\nif _failed:\n    print(f\"   Flagged files: {_failed}\")\n    print(f\"   These are still written to disk -- use the 'regenerate one file' utility cell at the end to fix them individually.\")\nprint(f\"   Full per-file report saved to {GENERATION_REPORT_PATH}\")\n"

_PIPELINE_CELL_16 = "TARGET_UNITY_VERSION = \"6000.0.35f1\"  # <-- change this to match your installed Editor version,\n                                        #     e.g. \"6000.5.4f1\", if it's different\n\nproject_settings_dir = os.path.join(OUTPUT_DIR, \"ProjectSettings\")\nos.makedirs(project_settings_dir, exist_ok=True)\n\n# The one file that actually matters for Hub's \"Restricted Editor Version\" detection.\nwith open(os.path.join(project_settings_dir, \"ProjectVersion.txt\"), \"w\") as f:\n    f.write(f\"m_EditorVersion: {TARGET_UNITY_VERSION}\\n\")\n    f.write(f\"m_EditorVersionWithRevision: {TARGET_UNITY_VERSION} (0000000000)\\n\")\n\n# A valid, empty scene list -- safe, standard boilerplate, avoids a separate first-open prompt\n# about no scenes being in the build settings.\neditor_build_settings = \"\"\"%YAML 1.1\n%TAG !u! tag:unity3d.com,2011:\n--- !u!1045 &1\nEditorBuildSettings:\n  m_ObjectHideFlags: 0\n  serializedVersion: 2\n  m_Scenes: []\n  m_configObjects: {}\n\"\"\"\nwith open(os.path.join(project_settings_dir, \"EditorBuildSettings.asset\"), \"w\") as f:\n    f.write(editor_build_settings)\n\n# Make sure Assets/ exists even if the plan generated nothing directly under it (e.g. everything\n# landed under Assets/_Project/ as the M1/M2 specs' folder convention expects).\nos.makedirs(os.path.join(OUTPUT_DIR, \"Assets\"), exist_ok=True)\n\nprint(f\"\\u2705 Wrote ProjectSettings/ProjectVersion.txt (version: {TARGET_UNITY_VERSION})\")\nprint(\"\\u2705 Wrote ProjectSettings/EditorBuildSettings.asset (empty scene list)\")\nprint(\"\\nEverything else under ProjectSettings/ will be created automatically by the Editor\")\nprint(\"the first time it opens this project -- that's normal, not a sign anything is missing.\")\n\n\n# Verify the structure is actually correct before handing off to the zip step --\n# catch a structural problem here, not later as a cryptic error in a different notebook.\nassets_path = os.path.join(OUTPUT_DIR, \"Assets\")\nassets_contents = []\nfor root, dirs, files in os.walk(assets_path):\n    for fn in files:\n        assets_contents.append(os.path.relpath(os.path.join(root, fn), OUTPUT_DIR))\n\nprint(f\"\\nAssets/ contains {len(assets_contents)} file(s):\")\nfor p in sorted(assets_contents)[:30]:\n    print(\" \", p)\nif len(assets_contents) > 30:\n    print(f\"  ... and {len(assets_contents) - 30} more\")\n\nif len(assets_contents) == 0:\n    print(\"\\n\\u26a0\\ufe0f WARNING: Assets/ is empty. Every planned file landed somewhere else --\")\n    print(\"check FILE_PLAN's paths above; something upstream of this step needs a re-run.\")\nelse:\n    print(\"\\n\\u2705 Assets/ has real content -- structure looks correct.\")\n"

_PIPELINE_CELL_18 = "# --- Feature 8: auto-generated project README, bundled into the zip ---\n_by_type = {}\nfor entry in FILE_PLAN:\n    _by_type.setdefault(entry[\"type\"], []).append(entry)\n\n_readme_lines = [\n    \"# Generated Unity Project\",\n    \"\",\n    f\"Generated by: {'Unity-tuned LoRA adapter' if USING_TUNED_MODEL else 'base model (no fine-tuning)'}\",\n    f\"Target Unity version: {TARGET_UNITY_VERSION}\",\n    f\"Total files: {len(FILE_PLAN)}\",\n    \"\",\n    \"## Files by type\",\n    \"\",\n]\nfor ftype, entries in sorted(_by_type.items()):\n    _readme_lines.append(f\"### {ftype} ({len(entries)})\")\n    for entry in entries:\n        status = GENERATION_REPORT.get(entry[\"path\"], {}).get(\"status\", \"unknown\")\n        flag = \" \\u26a0\\ufe0f flagged\" if status == \"failed\" else \"\"\n        _readme_lines.append(f\"- `{entry['path']}`{flag} -- {entry['description']}\")\n    _readme_lines.append(\"\")\n\n_readme_lines += [\n    \"## Before opening in Unity\",\n    \"\",\n    \"1. See `HUMAN_SETUP.md` (generated alongside this file) for what needs to be wired up \"\n    \"manually in the Editor -- scenes, prefabs, and ScriptableObject import are not auto-generated.\",\n    \"2. Check `generation_report.json` in the notebook's `logs/` output for any files that needed \"\n    \"a retry or are still flagged after validation.\",\n    \"3. Open this project folder directly in Unity Hub once extracted from the zip.\",\n]\n\nwith open(os.path.join(OUTPUT_DIR, \"README.md\"), \"w\", encoding=\"utf-8\") as f:\n    f.write(\"\\n\".join(_readme_lines))\n\nprint(f\"\\u2705 Wrote README.md summarizing {len(FILE_PLAN)} files across {len(_by_type)} type(s).\")\n"

_PIPELINE_CELL_20 = "import shutil\n\n# --- Feature 9: zip integrity check, BEFORE creating the zip ---\n# Verifies every planned file actually exists on disk, is non-empty, and isn't suspiciously\n# truncated (e.g. a generation call that returned almost nothing due to an upstream error that\n# didn't raise an exception) -- catches that here, with a clear list, instead of it surfacing\n# later as a mysteriously broken/incomplete download.\nMIN_FILE_BYTES = 20  # even a minimal valid file (an empty class, a tiny JSON object) clears this\n_missing, _empty, _tiny = [], [], []\nfor entry in FILE_PLAN:\n    fp = os.path.join(OUTPUT_DIR, entry[\"path\"])\n    if not os.path.exists(fp):\n        _missing.append(entry[\"path\"])\n        continue\n    size = os.path.getsize(fp)\n    if size == 0:\n        _empty.append(entry[\"path\"])\n    elif size < MIN_FILE_BYTES:\n        _tiny.append((entry[\"path\"], size))\n\nif _missing or _empty or _tiny:\n    print(\"\\u26a0\\ufe0f  Integrity check found issues before zipping:\")\n    if _missing:\n        print(f\"  Missing entirely ({len(_missing)}): {_missing}\")\n    if _empty:\n        print(f\"  Empty (0 bytes) ({len(_empty)}): {_empty}\")\n    if _tiny:\n        print(f\"  Suspiciously small (<{MIN_FILE_BYTES}B) ({len(_tiny)}): {_tiny}\")\n    print(\"  Zipping anyway -- these are still worth checking with the 'regenerate one file' \"\n          \"utility (Step 9) or the Step 8 validation pass after this cell.\")\nelse:\n    print(f\"\\u2705 Integrity check passed -- all {len(FILE_PLAN)} planned files present, non-empty, and reasonably sized.\")\n\nZIP_BASENAME = os.path.join(WORKDIR, \"generated_project\")\nzip_path = shutil.make_archive(ZIP_BASENAME, \"zip\", OUTPUT_DIR)\n\nfinal_path = \"/kaggle/working/generated_project.zip\"\nif zip_path != final_path:\n    shutil.move(zip_path, final_path)\n\nsize_mb = os.path.getsize(final_path) / 1e6\nprint(f\"\\n\\u2705 Zip created: {final_path} ({size_mb:.2f} MB)\")\nprint(f\"Contains {len(FILE_PLAN)} generated files (plus README.md and HUMAN_SETUP.md).\")\nprint(\"\\nDownload it from Kaggle's Output panel (right sidebar) once this notebook run finishes,\")\nprint(\"or via the file browser at /kaggle/working/generated_project.zip while the session is live.\")\nprint(\"\\nContinue to Step 8 below for a full syntax/compile validation pass over everything in this zip.\")\n\nfrom IPython.display import FileLink\ndisplay(FileLink(final_path))\n"

_PIPELINE_CELL_22 = "import zipfile\n\n_TS_AVAILABLE = False\ntry:\n    from tree_sitter_languages import get_parser\n    _cs_parser = get_parser(\"c_sharp\")\n    _TS_AVAILABLE = True\n    print(\"\\u2705 Real C# parser (tree-sitter) available -- using it for syntax validation.\")\nexcept Exception as e:\n    print(f\"\\u26a0\\ufe0f  tree-sitter unavailable ({e}) -- falling back to structural-only \"\n          f\"validation (brace/paren/bracket balance). Less precise, but still real coverage of \"\n          f\"the most common failure mode (truncated/malformed generation).\")\n\ndef _find_ts_errors(tree):\n    \"\"\"Walk the parse tree and collect every ERROR / MISSING node -- these are tree-sitter's\n    genuine syntax-error markers, not a heuristic.\"\"\"\n    errors = []\n    def walk(node):\n        if node.type == \"ERROR\" or node.is_missing:\n            line = node.start_point[0] + 1\n            errors.append(f\"line {line}: syntax error near '{node.type}'\")\n        for child in node.children:\n            walk(child)\n    walk(tree.root_node)\n    return errors\n\ndef validate_cs_content(content):\n    if _TS_AVAILABLE:\n        tree = _cs_parser.parse(bytes(content, \"utf-8\"))\n        return _find_ts_errors(tree)\n    return _check_balance(content)  # fallback -- reuses Step 6's structural check\n\n# --- Feature 10: full syntax/compile validation, run against the actual ZIP contents ---\nVALIDATION_REPORT = {}\nwith zipfile.ZipFile(final_path) as zf:\n    names = zf.namelist()\n    cs_files = [n for n in names if n.endswith(\".cs\")]\n    json_files = [n for n in names if n.endswith(\".json\")]\n\n    for n in cs_files:\n        content = zf.read(n).decode(\"utf-8\", errors=\"replace\")\n        errors = validate_cs_content(content)\n        VALIDATION_REPORT[n] = {\"type\": \"csharp\", \"errors\": errors}\n\n    for n in json_files:\n        content = zf.read(n).decode(\"utf-8\", errors=\"replace\")\n        try:\n            _json.loads(content)\n            errors = []\n        except Exception as e:\n            errors = [str(e)]\n        VALIDATION_REPORT[n] = {\"type\": \"json\", \"errors\": errors}\n\n_failed_validation = {n: r for n, r in VALIDATION_REPORT.items() if r[\"errors\"]}\n_total_checked = len(VALIDATION_REPORT)\n\nprint(f\"\\nValidated {len(cs_files)} .cs file(s) and {len(json_files)} .json file(s) from {final_path}\\n\")\nif _failed_validation:\n    print(f\"\\u274c {len(_failed_validation)}/{_total_checked} file(s) FAILED validation:\\n\")\n    for n, r in _failed_validation.items():\n        print(f\"  {n} ({r['type']}):\")\n        for err in r[\"errors\"][:5]:\n            print(f\"      {err}\")\n        if len(r[\"errors\"]) > 5:\n            print(f\"      ... and {len(r['errors']) - 5} more\")\nelse:\n    print(f\"\\u2705 All {_total_checked} checked file(s) passed validation with no errors.\")\n\nVALIDATION_REPORT_PATH = os.path.join(WORKDIR, \"validation_report.json\")\nwith open(VALIDATION_REPORT_PATH, \"w\", encoding=\"utf-8\") as f:\n    _json.dump(VALIDATION_REPORT, f, indent=2)\nprint(f\"\\nFull report saved to {VALIDATION_REPORT_PATH}\")\n\n# --- Feature 11: post-run summary statistics ---\n_total_lines = 0\n_total_chars = 0\nfor root, _, files in os.walk(OUTPUT_DIR):\n    for fn in files:\n        fp = os.path.join(root, fn)\n        try:\n            with open(fp, \"r\", encoding=\"utf-8\", errors=\"ignore\") as f:\n                text = f.read()\n            _total_lines += text.count(\"\\n\") + 1\n            _total_chars += len(text)\n        except Exception:\n            pass\n\n_pass_rate = 100.0 * (_total_checked - len(_failed_validation)) / _total_checked if _total_checked else 100.0\nprint(\"\\n--- Project summary ---\")\nprint(f\"  Model used:            {'Unity-tuned adapter' if USING_TUNED_MODEL else 'base model (no tuning)'}\")\nprint(f\"  Files planned:         {len(FILE_PLAN)}\")\nprint(f\"  Total lines written:   {_total_lines:,}\")\nprint(f\"  Total characters:      {_total_chars:,}\")\nprint(f\"  Generation wall-time:  {_gen_elapsed:.0f}s (this run's generation loop only)\")\nprint(f\"  Syntax validation:     {_total_checked - len(_failed_validation)}/{_total_checked} passed ({_pass_rate:.1f}%)\")\nprint(f\"  Files needing retry:   {len(_retried)}\")\nprint(f\"  Files still flagged:   {len(_failed)}\")\n"

_PIPELINE_CELLS_IN_ORDER = [_PIPELINE_CELL_12, _PIPELINE_CELL_14, _PIPELINE_CELL_16,
                            _PIPELINE_CELL_18, _PIPELINE_CELL_20, _PIPELINE_CELL_22]

_jobs = {}          # job_id -> {"state", "type", "error", "result_path" or "result_data"}
_job_queue = queue.Queue()
JOBS_DIR = os.path.join(BASE_DIR if "BASE_DIR" in globals() else WORKDIR, "jobs")
os.makedirs(JOBS_DIR, exist_ok=True)

def _combine_docs_zip(zip_bytes):
    extract_dir = os.path.join(WORKDIR, "spec_docs_job")
    if os.path.isdir(extract_dir):
        _shutil2.rmtree(extract_dir)
    os.makedirs(extract_dir, exist_ok=True)
    with _zf2.ZipFile(io.BytesIO(zip_bytes)) as zf:
        zf.extractall(extract_dir)
    priority = ["README.md", "Architecture.md", "ScriptsIndex.md", "ProjectVersion.txt",
                "Packages.md", "Scenes.md", "Prefabs.md", "Dependencies.md", "CodingGuidelines.md"]
    files = {f: os.path.join(extract_dir, f) for f in os.listdir(extract_dir)
             if f.endswith((".md", ".txt"))}
    ordered = [f for f in priority if f in files] + sorted(f for f in files if f not in priority)
    parts = []
    for fname in ordered:
        with open(files[fname], encoding="utf-8", errors="ignore") as fh:
            parts.append(f"=== {fname} ===\n\n{fh.read().strip()}")
    return "\n\n\n".join(parts)

def _run_generate_job(job_id, spec_text):
    global SPEC_TEXT
    SPEC_TEXT = spec_text
    job_ns = globals()
    for cell_src in _PIPELINE_CELLS_IN_ORDER:
        exec(compile(cell_src, "<pipeline_cell>", "exec"), job_ns)
    job_zip_path = os.path.join(JOBS_DIR, f"{job_id}.zip")
    _shutil2.copy(job_ns["final_path"], job_zip_path)
    return job_zip_path

def _run_repair_job(job_id, zip_bytes, errors_report, general_instructions):
    repaired_bytes, summary = repair_project(zip_bytes, errors_report, general_instructions)
    job_zip_path = os.path.join(JOBS_DIR, f"{job_id}.zip")
    with open(job_zip_path, "wb") as f:
        f.write(repaired_bytes)
    summary_path = os.path.join(JOBS_DIR, f"{job_id}.summary.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f)
    return job_zip_path

def _worker_loop():
    while True:
        job_id, task = _job_queue.get()
        try:
            _jobs[job_id]["state"] = "running"
            _jobs[job_id]["started_at"] = time.time()
            if task["type"] == "generate":
                result_path = _run_generate_job(job_id, task["spec_text"])
            else:
                result_path = _run_repair_job(job_id, task["zip_bytes"], task["errors_report"],
                                               task["general_instructions"])
            _jobs[job_id]["state"] = "done"
            _jobs[job_id]["result_path"] = result_path
        except Exception as e:
            _jobs[job_id]["state"] = "error"
            _jobs[job_id]["error"] = f"{type(e).__name__}: {e}\n{traceback.format_exc()[-2000:]}"
        _jobs[job_id]["finished_at"] = time.time()
        _job_queue.task_done()

threading.Thread(target=_worker_loop, daemon=True).start()

app2 = Flask("unified_server")

API_KEY_UNIFIED = os.environ.get("API_KEY", "dev-only-change-me")
try:
    from kaggle_secrets import UserSecretsClient as _USC2
    API_KEY_UNIFIED = _USC2().get_secret("API_KEY")
except Exception:
    print("No API_KEY secret found -- using dev fallback. Set a real one in Add-ons -> Secrets "
          "before exposing this publicly via ngrok.")

def _check_auth():
    auth = request.headers.get("Authorization", "")
    key = auth[7:] if auth.startswith("Bearer ") else request.headers.get("X-API-Key")
    return key == API_KEY_UNIFIED

@app2.route("/generate", methods=["POST"])
def http_generate():
    if not _check_auth():
        return jsonify({"error": "unauthorized"}), 401
    if "docs" not in request.files:
        return jsonify({"error": "missing 'docs' multipart field (a .zip of spec docs)"}), 400
    spec_text = _combine_docs_zip(request.files["docs"].read())
    job_id = str(uuid.uuid4())
    _jobs[job_id] = {"state": "queued", "type": "generate", "error": None, "result_path": None,
                      "queued_at": time.time(), "started_at": None, "finished_at": None}
    _job_queue.put((job_id, {"type": "generate", "spec_text": spec_text}))
    return jsonify({"job_id": job_id, "state": "queued"})

@app2.route("/repair", methods=["POST"])
def http_repair():
    if not _check_auth():
        return jsonify({"error": "unauthorized"}), 401
    if "project" not in request.files or "errors" not in request.files:
        return jsonify({"error": "missing 'project' and/or 'errors' multipart field"}), 400
    zip_bytes = request.files["project"].read()
    try:
        errors_report = json.loads(request.files["errors"].read())
    except json.JSONDecodeError as e:
        return jsonify({"error": f"CompilerErrors.json did not parse: {e}"}), 400
    general_instructions = "Fix the listed compiler errors."
    if "instructions" in request.files:
        general_instructions = request.files["instructions"].read().decode("utf-8", errors="ignore")
    job_id = str(uuid.uuid4())
    _jobs[job_id] = {"state": "queued", "type": "repair", "error": None, "result_path": None,
                      "queued_at": time.time(), "started_at": None, "finished_at": None}
    _job_queue.put((job_id, {"type": "repair", "zip_bytes": zip_bytes,
                              "errors_report": errors_report,
                              "general_instructions": general_instructions}))
    return jsonify({"job_id": job_id, "state": "queued"})

@app2.route("/status/<job_id>", methods=["GET"])
def http_status(job_id):
    if not _check_auth():
        return jsonify({"error": "unauthorized"}), 401
    job = _jobs.get(job_id)
    if not job:
        return jsonify({"error": "unknown job_id"}), 404
    return jsonify({"job_id": job_id, "state": job["state"], "type": job["type"], "error": job["error"]})

@app2.route("/result/<job_id>", methods=["GET"])
def http_result(job_id):
    if not _check_auth():
        return jsonify({"error": "unauthorized"}), 401
    job = _jobs.get(job_id)
    if not job or job["state"] != "done":
        return jsonify({"error": "job not done (or unknown)"}), 400
    resp = send_file(job["result_path"], mimetype="application/zip",
                      as_attachment=True, download_name="result.zip")
    summary_path = os.path.join(JOBS_DIR, f"{job_id}.summary.json")
    if job["type"] == "repair" and os.path.exists(summary_path):
        with open(summary_path) as f:
            resp.headers["X-Repair-Summary"] = f.read()
    return resp

@app2.route("/health", methods=["GET"])
def http_health():
    # Deliberately UNAUTHENTICATED -- this is what workflow 1's "wait for API" retry loop polls
    # before the notebook has necessarily finished starting up. No sensitive info in the response.
    return jsonify({"status": "ok"})

_DASHBOARD_HTML = """
<!DOCTYPE html><html><head><title>Generator/Repair Server</title>
<meta http-equiv="refresh" content="5">
<style>
body { font-family: monospace; background: #111; color: #ddd; padding: 20px; }
h1 { color: #7fd; }
table { border-collapse: collapse; width: 100%; margin-top: 12px; }
td, th { border: 1px solid #444; padding: 6px 10px; text-align: left; font-size: 13px; }
.state-done { color: #7f7; } .state-error { color: #f77; }
.state-running { color: #ff7; } .state-queued { color: #aaf; }
.err { color: #f99; white-space: pre-wrap; font-size: 11px; max-width: 500px; }
</style></head><body>
<h1>Generator / Repair server -- live activity</h1>
<p>Auto-refreshes every 5s. {job_count} job(s) total.</p>
<table>
<tr><th>Job ID</th><th>Type</th><th>State</th><th>Queued</th><th>Started</th><th>Finished</th><th>Duration</th><th>Error</th></tr>
{rows}
</table>
</body></html>
"""

def _fmt_time(t):
    return time.strftime("%H:%M:%S", time.localtime(t)) if t else "-"

@app2.route("/", methods=["GET"])
def http_dashboard():
    # This IS the "watch what's happening" view -- open the ngrok URL directly in a browser
    # (no Authorization header needed for this page specifically) to see every generate/repair
    # request as it's queued, picked up, and finished, live.
    rows = []
    for job_id, job in sorted(_jobs.items(), key=lambda kv: kv[1].get("queued_at") or 0, reverse=True):
        duration = ""
        if job.get("started_at") and job.get("finished_at"):
            duration = f"{job['finished_at'] - job['started_at']:.0f}s"
        elif job.get("started_at"):
            duration = f"{time.time() - job['started_at']:.0f}s (running)"
        err = (job.get("error") or "")[:300]
        rows.append(
            f"<tr><td>{job_id[:8]}</td><td>{job.get('type','?')}</td>"
            f"<td class='state-{job.get('state')}'>{job.get('state')}</td>"
            f"<td>{_fmt_time(job.get('queued_at'))}</td>"
            f"<td>{_fmt_time(job.get('started_at'))}</td>"
            f"<td>{_fmt_time(job.get('finished_at'))}</td>"
            f"<td>{duration}</td><td class='err'>{err}</td></tr>"
        )
    html = _DASHBOARD_HTML.format(job_count=len(_jobs), rows="\n".join(rows) or "<tr><td colspan=8>No jobs yet.</td></tr>")
    return html

def _run_flask2():
    app2.run(host="0.0.0.0", port=5001, use_reloader=False)

threading.Thread(target=_run_flask2, daemon=True).start()
print("Unified generate+repair job server starting on port 5001...")


In [ ]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient as _USC3
    NGROK_AUTH_TOKEN = _USC3().get_secret("NGROK_AUTH_TOKEN")
    if NGROK_AUTH_TOKEN:
        print("\u2705 NGROK_AUTH_TOKEN loaded from Kaggle Secrets.")
except Exception:
    pass

if not NGROK_AUTH_TOKEN:
    # Loud and explicit now, instead of silently proceeding with no token and printing one
    # easy-to-miss warning line -- that silence was exactly what made it unclear whether this
    # was ever being picked up at all.
    print(f"\n{'='*70}")
    print("NGROK_AUTH_TOKEN not found in Kaggle Secrets (Add-ons -> Secrets).")
    print("Get one free at https://dashboard.ngrok.com/get-started/your-authtoken")
    print(f"{'='*70}")
    NGROK_AUTH_TOKEN = input("Paste your ngrok auth token now (or leave blank to abort): ").strip()
    if not NGROK_AUTH_TOKEN:
        raise RuntimeError("An ngrok auth token is required to open the tunnel -- cannot continue.")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# With a free static ngrok domain, use it explicitly so the GitHub secret never needs updating:
#   public_url2 = ngrok.connect(5001, domain="your-static-domain.ngrok-free.app")
public_url2 = ngrok.connect(5001)
print("=" * 70)
print(f"Unified server is live at: {public_url2}")
print("Set this as NGROK_URL in your GitHub repo's Actions secrets (used by BOTH workflows).")
print("=" * 70)

import time as _t3
try:
    while True:
        _t3.sleep(60)
        print("[heartbeat] server still up:", public_url2)
except KeyboardInterrupt:
    print("Stopped.")


## Step 2b — Manual upload widget (optional, skip for automation)

Only needed if you want to test generation manually, one-off, inside this notebook -- blocks waiting for a file. Automated runs use Step 10a/10b instead and never need this cell.

In [ ]:
def _wait_for(condition_fn, timeout_s=3600, poll_msg="Waiting..."):
    """Block without freezing ipywidgets -- a plain sleep loop would starve the kernel's message
    queue, so the upload widget's callback (which is what actually saves the file) would never run.
    Pumping the kernel's event loop each tick keeps widget messages flowing during the wait."""
    kernel = get_ipython().kernel
    start = time.time()
    while not condition_fn():
        if time.time() - start > timeout_s:
            raise TimeoutError(f"Timed out. {poll_msg}")
        kernel.do_one_iteration()
        time.sleep(0.2)
        print(poll_msg, end="\r")
    print(poll_msg, "done.")

spec_upload = widgets.FileUpload(accept=".md", multiple=False, description="Upload spec.md")
spec_status = widgets.HTML(value="<i>Waiting for upload...</i>")
display(widgets.VBox([widgets.HTML("<b>Upload your spec .md file</b>"), spec_upload, spec_status]))

def _on_spec_upload(change):
    if not spec_upload.value:
        return
    item = spec_upload.value[0] if isinstance(spec_upload.value, (list, tuple)) else list(spec_upload.value.values())[0]
    with open(SPEC_PATH, "wb") as f:
        f.write(bytes(item["content"]))
    spec_status.value = f"\u2705 Saved {item.get('name','spec.md')} ({len(item['content'])/1024:.1f} KB)"

spec_upload.observe(_on_spec_upload, names="value")

# Blocking wait, same cell as the widget, so "Run All" genuinely sits here until you upload.
_wait_for(lambda: os.path.exists(SPEC_PATH), poll_msg="Waiting for spec.md upload...")

with open(SPEC_PATH, encoding="utf-8") as f:
    SPEC_TEXT = f.read()
print(f"\nLoaded spec: {len(SPEC_TEXT)} characters, {len(SPEC_TEXT.split())} words.")


## Step 5 — Planning pass: turn the spec into a concrete file list

One generation call, asking the model to read the whole spec and output a JSON plan of every file
to generate — scoped deliberately to what a text-only model can get right (see the top note).

In [ ]:
PLANNING_SYSTEM_PROMPT = """You are a senior Unity/C# engineer turning a written technical \
specification into a concrete file plan for a Unity project.

Only plan files you can write correctly as plain text with no Unity Editor available:
- C# scripts (MonoBehaviours and plain classes)
- JSON data files (for content that would normally be ScriptableObject assets -- a human will \
import these into real ScriptableObjects later; do NOT plan .asset files yourself)
- Packages/manifest.json
- Plain text project settings notes
- Exactly one HUMAN_SETUP.md as the last file, explaining what a human must do in the Unity \
Editor to finish wiring this into a working project (scenes, prefabs, importing the JSON data \
into real ScriptableObjects, NavMesh baking, etc.)

Do NOT plan .unity, .prefab, or .asset files -- those need Unity's Editor to keep GUID/fileID \
references valid and will be corrupted if hand-written as raw text.

MANDATORY -- you MUST include exactly one file at "Assets/Editor/ProjectSetup/SetupRunner.cs".
This is the single most important file in the whole plan. It is an Editor-only script (wrap the
whole class in #if UNITY_EDITOR) with a [MenuItem("Tools/Run Project Setup")] method that uses
Unity's real Editor APIs -- AssetDatabase, EditorSceneManager, PrefabUtility, GameObject,
AddComponent<T>() -- to programmatically build the scene and attach the generated scripts to
GameObjects, instead of a human doing it by hand. This is what turns generated CODE into an
actually-wired-up scene.

Small example of the shape a SetupRunner takes (yours must be specific to THIS project's actual
scripts and scene needs, not a copy of this):
```csharp
#if UNITY_EDITOR
using UnityEditor;
using UnityEditor.SceneManagement;
using UnityEngine;

public static class SetupRunner
{
    [MenuItem("Tools/Run Project Setup")]
    public static void Run()
    {
        var scene = EditorSceneManager.NewScene(NewSceneSetup.DefaultGameObjects);

        var player = new GameObject("Player");
        player.AddComponent<PlayerMovement>();
        player.AddComponent<CharacterController>();

        var gameManagerGO = new GameObject("GameManager");
        gameManagerGO.AddComponent<GameManager>();

        EditorSceneManager.SaveScene(scene, "Assets/_Project/Scenes/Main.unity");
        Debug.Log("Project setup complete.");
    }
}
#endif
```

CRITICAL path rule: every "path" value MUST start with the literal prefix "Assets/" or
"Packages/" -- for example "Assets/_Project/Scripts/Data/ProductData.cs", never just
"_Project/Scripts/Data/ProductData.cs" or "Scripts/Data/ProductData.cs". A Unity project
is only valid if its actual content lives inside a real Assets/ folder at the project
root -- omitting that prefix will produce a broken, unopenable project.

Output ONLY a JSON array, no other text, no markdown code fences. Each element:
{"path": "Assets/... or Packages/...", "type": "csharp|json|manifest|text|readme", \
"description": "what this file must contain, specific enough to write it correctly in isolation"}

Plan AT MOST 35 files total, even if the specification describes more scope than that -- prefer \
fewer, more substantial files (e.g. one Combat/ folder of related scripts as 3-4 files, not 15 \
tiny ones) over an exhaustive breakdown. This plan must fit in a single response with no \
continuation needed.

Keep each "description" to ONE short, plain-language sentence (max ~20 words) describing WHAT \
the file does -- e.g. "ScriptableObject holding a product's name, price, and shelf category." \
Do NOT write actual C# code, attributes, field declarations, or syntax in the description -- that \
level of detail belongs in the file itself (Step 6 generates the real code from this description), \
not in the plan. Verbose code-like descriptions are the main reason planning runs out of its \
token budget before finishing the file list.
"""

PLANNING_SPEC_MAX_CHARS = 6000  # lowered from 16000 -- your spec (15,903 chars) was UNDER the
# old cap, so it never truncated, and still OOM'd. If attention falls back to the memory-heavy
# "math" SDPA kernel (see Step 4's new kernel-forcing attempt), its memory scales roughly with
# the SQUARE of sequence length -- cutting input length is the most reliable lever available,
# more reliable than hoping the kernel-forcing trick above actually engages on this GPU/torch combo.
_spec_for_planning = SPEC_TEXT[:PLANNING_SPEC_MAX_CHARS]
if len(SPEC_TEXT) > PLANNING_SPEC_MAX_CHARS:
    _spec_for_planning += ("\n\n[...spec truncated for length -- plan based on what's above; "
                            "per-file generation later still sees the plan's own 'description' field...]")
    print(f"\u26a0\ufe0f  spec.md is {len(SPEC_TEXT)} chars -- truncated to {PLANNING_SPEC_MAX_CHARS} "
          f"for the planning prompt to avoid a prefill OOM. If the plan looks incomplete, this is why; "
          f"raise PLANNING_SPEC_MAX_CHARS above if you have GPU headroom to spare.")

planning_user_prompt = f"Specification:\n\n{_spec_for_planning}\n\nProduce the file plan JSON now."

_log(f"Starting planning pass... ({'Unity-tuned' if USING_TUNED_MODEL else 'BASE (untuned)'} model)")
# force_offload_cache removed -- the 2-GPU split in Step 3 is the real fix for memory pressure
# now, and the offloaded-cache code path was implicated in a run that produced degenerate,
# repetitive garbage output instead of crashing. repetition_penalty in Step 4 is a second
# guard against that specific failure mode regardless of cause.
# max_continuations=0 is deliberate: a continuation re-tokenizes the ENTIRE conversation so
# far (including the full first call's output) as input to the next call -- for planning, that
# meant a second call starting from ~5,900+ tokens of context, which is exactly what triggered
# the OOM (not GPU count -- a single attention op still runs on one GPU either way). Better to
# stop cleanly here and let the JSON-repair/retry logic below handle a truncated result than to
# pay for an expensive, OOM-prone continuation. The 35-file cap above should make hitting the
# length ceiling at all much less likely in the first place.
plan_raw = generate_long(PLANNING_SYSTEM_PROMPT, planning_user_prompt,
                          max_new_tokens=6000, max_continuations=0, temperature=PLANNING_TEMPERATURE)
                          # 6000, up from 4096 -- a single (non-continuation) call at this length
                          # did NOT OOM last run, it just ran out of budget mid-file. More headroom
                          # directly reduces truncation risk; still bounded, still no continuation.

# The model may still wrap the JSON in fences despite instructions -- strip defensively.
plan_clean = plan_raw.strip()
if plan_clean.startswith("```"):
    plan_clean = plan_clean.split("```")[1]
    if plan_clean.startswith("json"):
        plan_clean = plan_clean[4:]
plan_clean = plan_clean.strip()

def _try_parse_json(text):
    try:
        return _json.loads(text), None
    except _json.JSONDecodeError as e:
        return None, e

FILE_PLAN, _err = _try_parse_json(plan_clean)

if FILE_PLAN is None:
    # Common, cheaply-fixable glitches: trailing commas before ] or }, and single quotes
    # instead of double quotes. Try each repair in turn before giving up.
    _repaired = re.sub(r',(\s*[\]\}])', r'\1', plan_clean)  # strip trailing commas
    FILE_PLAN, _err2 = _try_parse_json(_repaired)
    if FILE_PLAN is None:
        _repaired2 = _repaired.replace("'", '"')
        FILE_PLAN, _err3 = _try_parse_json(_repaired2)

if FILE_PLAN is None:
    # Last resort: the array may just be genuinely truncated mid-object (ran out of token
    # budget), not malformed -- salvage every COMPLETE top-level object before the cutoff and
    # close the array, rather than discarding an otherwise-good partial plan entirely.
    _last_complete = plan_clean.rfind("},")
    if _last_complete != -1:
        _salvaged = plan_clean[:_last_complete + 1].rstrip().rstrip(",") + "\n]"
        FILE_PLAN, _err4 = _try_parse_json(_salvaged)
        if FILE_PLAN is not None:
            print(f"\u26a0\ufe0f  Plan was truncated mid-file -- salvaged {len(FILE_PLAN)} complete "
                  f"file entries before the cutoff and dropped the incomplete last one. If this is "
                  f"fewer files than your spec needs, re-run this cell (sampling varies) or raise "
                  f"max_new_tokens on the generate_long call above.")

if FILE_PLAN is None:
    print(f"\u274c Planning JSON still invalid after repair attempts: {_err}")
    print(f"\n--- Raw model output (first 2000 chars) ---\n{plan_raw[:2000]}\n--- end ---\n")
    print("Common fix: just re-run this cell -- planning is sampled, so a retry often produces "
          "valid JSON even when this one didn't. If it keeps failing, lower PLANNING_TEMPERATURE "
          "in this cell towards 0 for more deterministic (and more reliably well-formed) output.")
    raise _err

# Force-guarantee SetupRunner exists, rather than just hoping the prompt instruction was
# followed -- this is the most important file in the plan, so it doesn't get left to chance.
_SETUPRUNNER_PATH = "Assets/Editor/ProjectSetup/SetupRunner.cs"
if not any(e["path"] == _SETUPRUNNER_PATH for e in FILE_PLAN):
    print(f"\u26a0\ufe0f  Model omitted SetupRunner.cs despite the mandatory instruction -- "
          f"injecting it into the plan directly rather than leaving it out.")
    FILE_PLAN.append({
        "path": _SETUPRUNNER_PATH,
        "type": "csharp",
        "description": (
            "Editor-only script (#if UNITY_EDITOR) with a [MenuItem] method that builds the "
            "main scene and attaches every generated MonoBehaviour script planned above to an "
            "appropriate GameObject using AddComponent<T>(), then saves the scene. This is what "
            "wires the generated code into an actually-usable scene instead of leaving it "
            "disconnected."
        ),
    })

print(f"\n\u2705 Planned {len(FILE_PLAN)} files:\n")
for entry in FILE_PLAN:
    print(f"  - {entry['path']}  ({entry['type']})")

# --- Feature 5: duplicate / case-insensitive filename collision detection ---
# Unity projects commonly get imported on both case-sensitive (Linux/Mac) and case-insensitive
# (Windows) filesystems. Two planned files that differ only by case (e.g. "PlayerData.cs" and
# "playerdata.cs") would silently overwrite one another on Windows, or worse, produce genuinely
# undefined behavior on Unity's asset database. Catch that here, before any generation happens,
# rather than discovering it as a mysteriously missing file later.
_seen_lower = {}
_collisions = []
for entry in FILE_PLAN:
    key = entry["path"].lower()
    if key in _seen_lower:
        _collisions.append((_seen_lower[key], entry["path"]))
    else:
        _seen_lower[key] = entry["path"]

if _collisions:
    print(f"\n\u26a0\ufe0f  {len(_collisions)} case-insensitive filename collision(s) found in the plan:")
    for a, b in _collisions:
        print(f"    {a!r}  <->  {b!r}")
    print("    These will overwrite each other on case-insensitive filesystems (Windows). "
          "Consider editing FILE_PLAN by hand before continuing, or re-running this cell.")
else:
    print("\n\u2705 No filename collisions in the plan.")


## Step 6 — Generation loop: write every planned file, uncapped, in full

Files are generated in **batches** (`BATCH_SIZE`, set in Step 4) rather than one at a time — since
this model has to be split across both T4s just to fit, a single request is inherently sequential
between the two halves, so batching multiple files into one call is the real lever for keeping both
GPUs busier at once. Watch `nvidia-smi -l 1` in a terminal (or Kaggle's GPU usage graph) during a
run to see the effect, and adjust `BATCH_SIZE` up or down based on what you see and whether you hit
memory pressure. Any file that hits its length cap mid-batch automatically falls back to a solo
continuation call (from Step 4's auto-continue logic) to finish it properly, rather than being cut
short. Progress is checkpointed to disk — if the Kaggle session dies partway through, re-running
this cell skips files that already exist and picks up where it left off.

In [ ]:
GENERATION_SYSTEM_PROMPT = """You are a senior Unity/C# engineer. Write ONE complete, real, \
compiling file -- never a stub, never a placeholder, never a TODO, never truncated. Take as much \
space as the file genuinely needs to be correct and complete; do not artificially shorten it. \
Output ONLY the raw file content -- no markdown code fences, no explanation before or after, no \
commentary. If writing C#, include using statements, full method bodies, and real logic matching \
the description exactly."""

def _normalize_path(entry):
    p = entry["path"].lstrip("/")
    if not (p.startswith("Assets/") or p.startswith("Packages/")):
        p = f"Assets/{p}"
        entry["path"] = p
    return p

def _build_user_prompt(entry, extra_instruction=None):
    SPEC_EXCERPT_MAX_CHARS = 4000
    spec_for_prompt = SPEC_TEXT[:SPEC_EXCERPT_MAX_CHARS]
    if len(SPEC_TEXT) > SPEC_EXCERPT_MAX_CHARS:
        spec_for_prompt += "\n\n[...spec truncated for length -- see the description field below for this file's specific requirements...]"
    prompt = (
        f"Specification excerpt, for background context:\n\n{spec_for_prompt}\n\n"
        f"---\n\nNow write this specific file in full:\n\n"
        f"Path: {entry['path']}\nType: {entry['type']}\nRequired contents: {entry['description']}\n\n"
        f"Write the complete file content now."
    )
    if extra_instruction:
        prompt += f"\n\nIMPORTANT -- fix this specific problem from the previous attempt: {extra_instruction}"
    return prompt

def _clean_fences(content):
    content_clean = content.strip()
    if content_clean.startswith("```"):
        lines = content_clean.split("\n")
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        content_clean = "\n".join(lines)
    return content_clean

# --- Feature 2: fast structural balance check (catches truncation/malformed output cheaply) ---
def _check_balance(content):
    problems = []
    pairs = [("{", "}"), ("(", ")"), ("[", "]")]
    for open_ch, close_ch in pairs:
        o, c = content.count(open_ch), content.count(close_ch)
        if o != c:
            problems.append(f"unbalanced '{open_ch}{close_ch}': {o} open vs {c} close")
    return problems
    # Note: this is a heuristic, not a real parser -- a brace character inside a string literal
    # or comment can occasionally cause a false positive. Step 8's final pass uses a real parser
    # where available and is the authoritative check; this one exists to catch the common,
    # cheap-to-detect case (truncated/cut-off generation) immediately, without waiting until
    # the whole project is done.

# --- Feature 4: class-name-must-match-filename check (hard Unity requirement for MonoBehaviours) ---
_CLASS_RE = re.compile(r'public\s+(?:sealed\s+|abstract\s+|partial\s+)*(?:class|struct|interface)\s+(\w+)')

def _check_class_name(entry, content):
    if entry["type"] != "csharp":
        return []
    filename_stem = os.path.splitext(os.path.basename(entry["path"]))[0]
    matches = _CLASS_RE.findall(content)
    if len(matches) == 1 and matches[0] != filename_stem:
        return [f"public type '{matches[0]}' doesn't match filename '{filename_stem}' -- "
                f"Unity requires these to match exactly for MonoBehaviours to attach in the Editor"]
    return []

def _validate_file(entry, content):
    return _check_balance(content) + _check_class_name(entry, content)

def _write_file(entry, content):
    out_path = os.path.join(OUTPUT_DIR, entry["path"])
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    content_clean = _clean_fences(content)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(content_clean)
    _log(f"  -> {entry['path']}: {len(content_clean)} chars written.")
    return content_clean

# --- Feature 6: structured per-file generation report ---
GENERATION_REPORT_PATH = os.path.join(LOG_DIR, "generation_report.json")
GENERATION_REPORT = {}

def _generate_with_retries(entry, base_prompt, initial_content, initial_hit_cap):
    """Handles the length-cap-continuation case (existing behavior) AND validation-driven
    retries (Feature 3) -- regenerating a file up to MAX_FIX_ATTEMPTS times, feeding the
    specific detected problem back to the model each time, before giving up and flagging it."""
    if initial_hit_cap:
        content = generate_long(GENERATION_SYSTEM_PROMPT, base_prompt,
                                 max_new_tokens=MAX_NEW_TOKENS_PER_CALL,
                                 max_continuations=MAX_CONTINUATIONS, temperature=FILE_GEN_TEMPERATURE,
                                 seed_text=initial_content)
    else:
        content = initial_content

    attempts = 1
    issues = _validate_file(entry, _clean_fences(content))
    while issues and attempts <= MAX_FIX_ATTEMPTS:
        _log(f"  {entry['path']}: validation failed ({'; '.join(issues)}) -- retry {attempts}/{MAX_FIX_ATTEMPTS}...")
        retry_prompt = _build_user_prompt(entry, extra_instruction="; ".join(issues))
        content = generate_long(GENERATION_SYSTEM_PROMPT, retry_prompt,
                                 max_new_tokens=MAX_NEW_TOKENS_PER_CALL,
                                 max_continuations=MAX_CONTINUATIONS, temperature=FILE_GEN_TEMPERATURE)
        issues = _validate_file(entry, _clean_fences(content))
        attempts += 1

    status = "ok" if attempts == 1 and not issues else ("retried_ok" if not issues else "failed")
    GENERATION_REPORT[entry["path"]] = {"status": status, "attempts": attempts, "issues": issues}
    if issues:
        _log(f"  \u26a0\ufe0f {entry['path']}: still failing validation after {attempts} attempt(s): {'; '.join(issues)}")
    return content

# Skip anything already generated (checkpoint/resume behavior across session restarts).
pending = []
for entry in FILE_PLAN:
    _normalize_path(entry)
    out_path = os.path.join(OUTPUT_DIR, entry["path"])
    if os.path.exists(out_path):
        print(f"Skipping (already generated): {entry['path']}")
        GENERATION_REPORT.setdefault(entry["path"], {"status": "ok", "attempts": 0, "issues": []})
    else:
        pending.append(entry)

print(f"\n{len(pending)} file(s) to generate, {len(FILE_PLAN) - len(pending)} already done.\n")

_gen_start_time = time.time()

for batch_start in range(0, len(pending), BATCH_SIZE):
    batch = pending[batch_start: batch_start + BATCH_SIZE]
    _log(f"Generating batch of {len(batch)}: {[e['path'] for e in batch]}")

    user_prompts = [_build_user_prompt(entry) for entry in batch]
    batch_results = generate_batch(GENERATION_SYSTEM_PROMPT, user_prompts,
                                    max_new_tokens=MAX_NEW_TOKENS_PER_CALL, temperature=FILE_GEN_TEMPERATURE)

    for entry, prompt, (chunk, hit_cap) in zip(batch, user_prompts, batch_results):
        content = _generate_with_retries(entry, prompt, chunk, hit_cap)
        _write_file(entry, content)

_gen_elapsed = time.time() - _gen_start_time

with open(GENERATION_REPORT_PATH, "w", encoding="utf-8") as f:
    _json.dump(GENERATION_REPORT, f, indent=2)

_failed = [p for p, r in GENERATION_REPORT.items() if r["status"] == "failed"]
_retried = [p for p, r in GENERATION_REPORT.items() if r["status"] == "retried_ok"]
print(f"\n\u2705 All {len(FILE_PLAN)} files generated in {OUTPUT_DIR} ({_gen_elapsed:.0f}s this run)")
print(f"   {len(_retried)} needed a retry and passed after fixing; {len(_failed)} still flagged after {MAX_FIX_ATTEMPTS} attempts.")
if _failed:
    print(f"   Flagged files: {_failed}")
    print(f"   These are still written to disk -- use the 'regenerate one file' utility cell at the end to fix them individually.")
print(f"   Full per-file report saved to {GENERATION_REPORT_PATH}")


## Step 6.5 — Bootstrap a real Unity project scaffold

**Why this step exists:** Unity Hub couldn't detect an Editor version for the raw generator output
because the one file Hub actually reads for that — `ProjectSettings/ProjectVersion.txt` — never
existed; the generator only ever produced `Assets/` and `Packages/manifest.json`.

This is a different situation from scenes/prefabs (which this generator deliberately never
attempts — see the top note). `ProjectVersion.txt` is a plain two-line text file with no GUIDs or
cross-references to get wrong, so it's safe to generate directly rather than needing a real Editor
to produce it. **Every other `ProjectSettings/*.asset` file is genuinely optional** — Unity creates
each one with sane defaults automatically the first time the Editor opens a project that's missing
it, exactly like opening a brand-new empty project. So writing just this one file (plus an empty,
valid `EditorBuildSettings.asset`, since a missing scene list is a slightly more common source of
first-open confusion) is enough to make the output open cleanly, with no risk of the
GUID-corruption problem that ruled out generating scenes/prefabs in the first place.

**Set `TARGET_UNITY_VERSION` below to match whatever Editor version you actually have installed**
(check Unity Hub's Installs tab — the version in your screenshot was `6000.5.4f1`, which may differ
from this notebook's default).

In [ ]:
TARGET_UNITY_VERSION = "6000.0.35f1"  # <-- change this to match your installed Editor version,
                                        #     e.g. "6000.5.4f1", if it's different

project_settings_dir = os.path.join(OUTPUT_DIR, "ProjectSettings")
os.makedirs(project_settings_dir, exist_ok=True)

# The one file that actually matters for Hub's "Restricted Editor Version" detection.
with open(os.path.join(project_settings_dir, "ProjectVersion.txt"), "w") as f:
    f.write(f"m_EditorVersion: {TARGET_UNITY_VERSION}\n")
    f.write(f"m_EditorVersionWithRevision: {TARGET_UNITY_VERSION} (0000000000)\n")

# A valid, empty scene list -- safe, standard boilerplate, avoids a separate first-open prompt
# about no scenes being in the build settings.
editor_build_settings = """%YAML 1.1
%TAG !u! tag:unity3d.com,2011:
--- !u!1045 &1
EditorBuildSettings:
  m_ObjectHideFlags: 0
  serializedVersion: 2
  m_Scenes: []
  m_configObjects: {}
"""
with open(os.path.join(project_settings_dir, "EditorBuildSettings.asset"), "w") as f:
    f.write(editor_build_settings)

# Make sure Assets/ exists even if the plan generated nothing directly under it (e.g. everything
# landed under Assets/_Project/ as the M1/M2 specs' folder convention expects).
os.makedirs(os.path.join(OUTPUT_DIR, "Assets"), exist_ok=True)

print(f"\u2705 Wrote ProjectSettings/ProjectVersion.txt (version: {TARGET_UNITY_VERSION})")
print("\u2705 Wrote ProjectSettings/EditorBuildSettings.asset (empty scene list)")
print("\nEverything else under ProjectSettings/ will be created automatically by the Editor")
print("the first time it opens this project -- that's normal, not a sign anything is missing.")


# Verify the structure is actually correct before handing off to the zip step --
# catch a structural problem here, not later as a cryptic error in a different notebook.
assets_path = os.path.join(OUTPUT_DIR, "Assets")
assets_contents = []
for root, dirs, files in os.walk(assets_path):
    for fn in files:
        assets_contents.append(os.path.relpath(os.path.join(root, fn), OUTPUT_DIR))

print(f"\nAssets/ contains {len(assets_contents)} file(s):")
for p in sorted(assets_contents)[:30]:
    print(" ", p)
if len(assets_contents) > 30:
    print(f"  ... and {len(assets_contents) - 30} more")

if len(assets_contents) == 0:
    print("\n\u26a0\ufe0f WARNING: Assets/ is empty. Every planned file landed somewhere else --")
    print("check FILE_PLAN's paths above; something upstream of this step needs a re-run.")
else:
    print("\n\u2705 Assets/ has real content -- structure looks correct.")


## Step 6.6 — Auto-generate a project README

A deterministic, programmatically-generated `README.md` (not model-generated -- this just formats data already in `FILE_PLAN` and `GENERATION_REPORT`), summarizing what was generated, grouped by type, plus which model produced it.

In [ ]:
# --- Feature 8: auto-generated project README, bundled into the zip ---
_by_type = {}
for entry in FILE_PLAN:
    _by_type.setdefault(entry["type"], []).append(entry)

_readme_lines = [
    "# Generated Unity Project",
    "",
    f"Generated by: {'Unity-tuned LoRA adapter' if USING_TUNED_MODEL else 'base model (no fine-tuning)'}",
    f"Target Unity version: {TARGET_UNITY_VERSION}",
    f"Total files: {len(FILE_PLAN)}",
    "",
    "## Files by type",
    "",
]
for ftype, entries in sorted(_by_type.items()):
    _readme_lines.append(f"### {ftype} ({len(entries)})")
    for entry in entries:
        status = GENERATION_REPORT.get(entry["path"], {}).get("status", "unknown")
        flag = " \u26a0\ufe0f flagged" if status == "failed" else ""
        _readme_lines.append(f"- `{entry['path']}`{flag} -- {entry['description']}")
    _readme_lines.append("")

_readme_lines += [
    "## Before opening in Unity",
    "",
    "1. See `HUMAN_SETUP.md` (generated alongside this file) for what needs to be wired up "
    "manually in the Editor -- scenes, prefabs, and ScriptableObject import are not auto-generated.",
    "2. Check `generation_report.json` in the notebook's `logs/` output for any files that needed "
    "a retry or are still flagged after validation.",
    "3. Open this project folder directly in Unity Hub once extracted from the zip.",
]

with open(os.path.join(OUTPUT_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(_readme_lines))

print(f"\u2705 Wrote README.md summarizing {len(FILE_PLAN)} files across {len(_by_type)} type(s).")


## Step 7 — Zip the generated project and make it downloadable

In [ ]:
import shutil

# --- Feature 9: zip integrity check, BEFORE creating the zip ---
# Verifies every planned file actually exists on disk, is non-empty, and isn't suspiciously
# truncated (e.g. a generation call that returned almost nothing due to an upstream error that
# didn't raise an exception) -- catches that here, with a clear list, instead of it surfacing
# later as a mysteriously broken/incomplete download.
MIN_FILE_BYTES = 20  # even a minimal valid file (an empty class, a tiny JSON object) clears this
_missing, _empty, _tiny = [], [], []
for entry in FILE_PLAN:
    fp = os.path.join(OUTPUT_DIR, entry["path"])
    if not os.path.exists(fp):
        _missing.append(entry["path"])
        continue
    size = os.path.getsize(fp)
    if size == 0:
        _empty.append(entry["path"])
    elif size < MIN_FILE_BYTES:
        _tiny.append((entry["path"], size))

if _missing or _empty or _tiny:
    print("\u26a0\ufe0f  Integrity check found issues before zipping:")
    if _missing:
        print(f"  Missing entirely ({len(_missing)}): {_missing}")
    if _empty:
        print(f"  Empty (0 bytes) ({len(_empty)}): {_empty}")
    if _tiny:
        print(f"  Suspiciously small (<{MIN_FILE_BYTES}B) ({len(_tiny)}): {_tiny}")
    print("  Zipping anyway -- these are still worth checking with the 'regenerate one file' "
          "utility (Step 9) or the Step 8 validation pass after this cell.")
else:
    print(f"\u2705 Integrity check passed -- all {len(FILE_PLAN)} planned files present, non-empty, and reasonably sized.")

ZIP_BASENAME = os.path.join(WORKDIR, "generated_project")
zip_path = shutil.make_archive(ZIP_BASENAME, "zip", OUTPUT_DIR)

final_path = "/kaggle/working/generated_project.zip"
if zip_path != final_path:
    shutil.move(zip_path, final_path)

size_mb = os.path.getsize(final_path) / 1e6
print(f"\n\u2705 Zip created: {final_path} ({size_mb:.2f} MB)")
print(f"Contains {len(FILE_PLAN)} generated files (plus README.md and HUMAN_SETUP.md).")
print("\nDownload it from Kaggle's Output panel (right sidebar) once this notebook run finishes,")
print("or via the file browser at /kaggle/working/generated_project.zip while the session is live.")
print("\nContinue to Step 8 below for a full syntax/compile validation pass over everything in this zip.")

from IPython.display import FileLink
display(FileLink(final_path))


## Step 8 — Validate the generated project (syntax check + integrity report)

Runs a full syntax/compile-style validation over every generated `.cs` file (using a real C# parser where available, falling back to the structural checks from Step 6 if the parser dependency isn't available in this environment) and validates every `.json` file parses correctly. Reports pass/fail per file against the actual zip contents, plus overall project statistics.

In [ ]:
import zipfile

_TS_AVAILABLE = False
try:
    from tree_sitter_languages import get_parser
    _cs_parser = get_parser("c_sharp")
    _TS_AVAILABLE = True
    print("\u2705 Real C# parser (tree-sitter) available -- using it for syntax validation.")
except Exception as e:
    print(f"\u26a0\ufe0f  tree-sitter unavailable ({e}) -- falling back to structural-only "
          f"validation (brace/paren/bracket balance). Less precise, but still real coverage of "
          f"the most common failure mode (truncated/malformed generation).")

def _find_ts_errors(tree):
    """Walk the parse tree and collect every ERROR / MISSING node -- these are tree-sitter's
    genuine syntax-error markers, not a heuristic."""
    errors = []
    def walk(node):
        if node.type == "ERROR" or node.is_missing:
            line = node.start_point[0] + 1
            errors.append(f"line {line}: syntax error near '{node.type}'")
        for child in node.children:
            walk(child)
    walk(tree.root_node)
    return errors

def validate_cs_content(content):
    if _TS_AVAILABLE:
        tree = _cs_parser.parse(bytes(content, "utf-8"))
        return _find_ts_errors(tree)
    return _check_balance(content)  # fallback -- reuses Step 6's structural check

# --- Feature 10: full syntax/compile validation, run against the actual ZIP contents ---
VALIDATION_REPORT = {}
with zipfile.ZipFile(final_path) as zf:
    names = zf.namelist()
    cs_files = [n for n in names if n.endswith(".cs")]
    json_files = [n for n in names if n.endswith(".json")]

    for n in cs_files:
        content = zf.read(n).decode("utf-8", errors="replace")
        errors = validate_cs_content(content)
        VALIDATION_REPORT[n] = {"type": "csharp", "errors": errors}

    for n in json_files:
        content = zf.read(n).decode("utf-8", errors="replace")
        try:
            _json.loads(content)
            errors = []
        except Exception as e:
            errors = [str(e)]
        VALIDATION_REPORT[n] = {"type": "json", "errors": errors}

_failed_validation = {n: r for n, r in VALIDATION_REPORT.items() if r["errors"]}
_total_checked = len(VALIDATION_REPORT)

print(f"\nValidated {len(cs_files)} .cs file(s) and {len(json_files)} .json file(s) from {final_path}\n")
if _failed_validation:
    print(f"\u274c {len(_failed_validation)}/{_total_checked} file(s) FAILED validation:\n")
    for n, r in _failed_validation.items():
        print(f"  {n} ({r['type']}):")
        for err in r["errors"][:5]:
            print(f"      {err}")
        if len(r["errors"]) > 5:
            print(f"      ... and {len(r['errors']) - 5} more")
else:
    print(f"\u2705 All {_total_checked} checked file(s) passed validation with no errors.")

VALIDATION_REPORT_PATH = os.path.join(WORKDIR, "validation_report.json")
with open(VALIDATION_REPORT_PATH, "w", encoding="utf-8") as f:
    _json.dump(VALIDATION_REPORT, f, indent=2)
print(f"\nFull report saved to {VALIDATION_REPORT_PATH}")

# --- Feature 11: post-run summary statistics ---
_total_lines = 0
_total_chars = 0
for root, _, files in os.walk(OUTPUT_DIR):
    for fn in files:
        fp = os.path.join(root, fn)
        try:
            with open(fp, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()
            _total_lines += text.count("\n") + 1
            _total_chars += len(text)
        except Exception:
            pass

_pass_rate = 100.0 * (_total_checked - len(_failed_validation)) / _total_checked if _total_checked else 100.0
print("\n--- Project summary ---")
print(f"  Model used:            {'Unity-tuned adapter' if USING_TUNED_MODEL else 'base model (no tuning)'}")
print(f"  Files planned:         {len(FILE_PLAN)}")
print(f"  Total lines written:   {_total_lines:,}")
print(f"  Total characters:      {_total_chars:,}")
print(f"  Generation wall-time:  {_gen_elapsed:.0f}s (this run's generation loop only)")
print(f"  Syntax validation:     {_total_checked - len(_failed_validation)}/{_total_checked} passed ({_pass_rate:.1f}%)")
print(f"  Files needing retry:   {len(_retried)}")
print(f"  Files still flagged:   {len(_failed)}")


## Step 9 — Regenerate a single file (utility)

If Step 8 flagged a specific file (or you just want to redo one), regenerate just that one file without rerunning the whole pipeline. Set `TARGET_PATH` below to the exact path shown in `FILE_PLAN` / the validation report, then run this cell.

In [ ]:
# --- Feature 12: standalone single-file regeneration utility ---
TARGET_PATH = "Assets/_Project/Scripts/PLACEHOLDER.cs"  # <-- set this to the file you want to redo,
                                                          #     copy it exactly from FILE_PLAN or the
                                                          #     validation report above

_target_entry = next((e for e in FILE_PLAN if e["path"] == TARGET_PATH), None)
if _target_entry is None:
    print(f"\u274c '{TARGET_PATH}' not found in FILE_PLAN. Copy the exact path from the printed "
          f"plan or validation report above (paths are case-sensitive).")
else:
    print(f"Regenerating: {TARGET_PATH}  ({'Unity-tuned' if USING_TUNED_MODEL else 'BASE'} model)")
    _prompt = _build_user_prompt(_target_entry)
    _content = generate_long(GENERATION_SYSTEM_PROMPT, _prompt,
                              max_new_tokens=MAX_NEW_TOKENS_PER_CALL,
                              max_continuations=MAX_CONTINUATIONS, temperature=FILE_GEN_TEMPERATURE)
    _issues = _validate_file(_target_entry, _clean_fences(_content))
    _written = _write_file(_target_entry, _content)
    GENERATION_REPORT[TARGET_PATH] = {
        "status": "ok" if not _issues else "failed",
        "attempts": GENERATION_REPORT.get(TARGET_PATH, {}).get("attempts", 0) + 1,
        "issues": _issues,
    }
    with open(GENERATION_REPORT_PATH, "w", encoding="utf-8") as f:
        _json.dump(GENERATION_REPORT, f, indent=2)
    if _issues:
        print(f"\u26a0\ufe0f Still has issues after regeneration: {_issues}")
        print("Try again, or edit the file directly in the zip once downloaded.")
    else:
        print(f"\u2705 Regenerated cleanly, {len(_written)} chars written.")
    print("\nRe-run Step 7 (zip cell) to include this update in a fresh download.")


## Optional — Start over with a different spec

In [ ]:
for path in (SPEC_PATH, OUTPUT_DIR, os.path.join(WORKDIR, "generated_project.zip"),
             GENERATION_REPORT_PATH, os.path.join(WORKDIR, "validation_report.json")):
    if os.path.isdir(path):
        shutil.rmtree(path)
    elif os.path.exists(path):
        os.remove(path)
os.makedirs(OUTPUT_DIR, exist_ok=True)
GENERATION_REPORT.clear()  # in-memory report from this session, reset alongside the files
print("Cleared. Re-run from Step 2 to upload a new spec.")
